In [3]:
# ============================================================
# FREIGHT-MNet Processing Step 4 FIXED:
# Aggregate BTS FMI county-pair truck travel-time quantiles
# to FAF-pair truck travel-time quantiles.
#
# FIX:
#   Use explicit FMI columns:
#       Origin CTFIPS
#       Destination CTFIPS
#       Crossings / Movements
#       25th Percentile (minutes)
#       50th Percentile (minutes)
#       75th Percentile (minutes)
#
# This overwrites previous zero-row outputs.
# ============================================================

from pathlib import Path
import json
import re
import gc
import warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")

FMI_ROOT = DATA_ROOT / "02_fmi"

COUNTY_TO_FAF_PATH = (
    DATA_ROOT
    / "09_crosswalks"
    / "county_to_faf"
    / "county_to_faf.parquet"
)

OUT_ROOT = DATA_ROOT / "08_processed" / "fmi_faf_aggregated"
BY_YEAR_DIR = OUT_ROOT / "by_year"
METADATA_DIR = OUT_ROOT / "_metadata"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
BY_YEAR_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

ALL_FINAL_PARQUET = OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_all.parquet"
ALL_FINAL_CSV = OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_all.csv"

EAST_FINAL_PARQUET = OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.parquet"
EAST_FINAL_CSV = OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.csv"

SUMMARY_PATH = METADATA_DIR / "fmi_faf_aggregation_summary_fixed.json"
YEARLY_MANIFEST_PATH = METADATA_DIR / "fmi_faf_aggregation_yearly_manifest_fixed.csv"
SCHEMA_REPORT_PATH = METADATA_DIR / "fmi_detected_schema_by_year_fixed.csv"

YEARS = list(range(2018, 2025))

# Important: overwrite previous zero-row outputs.
FORCE_REPROCESS = True

# Optional: save huge mapped county-pair tables. Usually not needed.
SAVE_MAPPED_COUNTY_PAIR_TABLES = False
MAPPED_PAIR_DIR = DATA_ROOT / "09_crosswalks" / "fmi_county_to_faf_pair"
MAPPED_PAIR_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("FMI county-pair -> FAF-pair aggregation FIXED")
print("FMI_ROOT:", FMI_ROOT)
print("COUNTY_TO_FAF_PATH:", COUNTY_TO_FAF_PATH)
print("OUT_ROOT:", OUT_ROOT)
print("=" * 100)

if not COUNTY_TO_FAF_PATH.exists():
    raise FileNotFoundError(f"Missing county_to_faf crosswalk: {COUNTY_TO_FAF_PATH}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def normalize_colname(c: str) -> str:
    return "".join(ch.lower() for ch in str(c) if ch.isalnum())


def normalize_fips_series(s: pd.Series) -> pd.Series:
    out = s.astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.str.replace(r"[^0-9]", "", regex=True)
    out = out.where(out.str.len() > 0, pd.NA)
    return out.str.zfill(5)


def find_col(columns, candidates, required=True, label="column"):
    norm_map = {normalize_colname(c): c for c in columns}

    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    if required:
        raise RuntimeError(
            f"Could not detect {label}. "
            f"Candidates={candidates}. Available columns={list(columns)}"
        )

    return None


def get_file_columns(path: Path) -> list[str]:
    if path.suffix.lower() == ".parquet":
        return list(pq.ParquetFile(str(path)).schema.names)
    return list(pd.read_csv(path, nrows=0).columns)


def find_fmi_file(year: int) -> Path:
    year_dir = FMI_ROOT / str(year)

    if not year_dir.exists():
        raise FileNotFoundError(f"Missing FMI year directory: {year_dir}")

    parquet_files = sorted(year_dir.glob(f"*{year}*.parquet"))
    if parquet_files:
        return parquet_files[0]

    csv_files = sorted(year_dir.glob(f"*{year}*.csv"))
    if csv_files:
        return csv_files[0]

    raise FileNotFoundError(f"No FMI CSV/Parquet found for {year} in {year_dir}")


def detect_fmi_schema(path: Path, year: int) -> dict:
    columns = get_file_columns(path)

    origin_col = find_col(
        columns,
        [
            "Origin CTFIPS",
            "Origin CTFIPS Code",
            "Origin County FIPS",
            "Origin County FIPS Code",
            "origin_ctfips",
            "origin_county_fips",
        ],
        label="origin_county_fips",
    )

    dest_col = find_col(
        columns,
        [
            "Destination CTFIPS",
            "Destination CTFIPS Code",
            "Destination County FIPS",
            "Destination County FIPS Code",
            "destination_ctfips",
            "destination_county_fips",
            "dest_county_fips",
        ],
        label="destination_county_fips",
    )

    q25_col = find_col(
        columns,
        [
            "25th Percentile (minutes)",
            "25th Percentile",
            "travel_time_p25_minutes",
            "p25",
        ],
        label="q25",
    )

    q50_col = find_col(
        columns,
        [
            "50th Percentile (minutes)",
            "50th Percentile",
            "travel_time_p50_minutes",
            "p50",
            "median travel time",
        ],
        label="q50",
    )

    q75_col = find_col(
        columns,
        [
            "75th Percentile (minutes)",
            "75th Percentile",
            "travel_time_p75_minutes",
            "p75",
        ],
        label="q75",
    )

    weight_col = find_col(
        columns,
        [
            "Movements",
            "Crossings",
            "Observation Count Bin",
            "observation_count_bin",
            "observation_count",
            "count_bin",
        ],
        required=False,
        label="weight_or_count_bin",
    )

    return {
        "year": year,
        "path": str(path),
        "origin_col": origin_col,
        "dest_col": dest_col,
        "q25_col": q25_col,
        "q50_col": q50_col,
        "q75_col": q75_col,
        "weight_col": weight_col,
        "all_columns": "|".join(columns),
    }


def read_fmi_selected(path: Path, schema: dict) -> pd.DataFrame:
    usecols = [
        schema["origin_col"],
        schema["dest_col"],
        schema["q25_col"],
        schema["q50_col"],
        schema["q75_col"],
    ]

    if schema["weight_col"] is not None:
        usecols.append(schema["weight_col"])

    usecols = list(dict.fromkeys(usecols))

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, columns=usecols)

    return pd.read_csv(path, usecols=usecols, low_memory=False)


def representative_count_weight_scalar(x) -> float:
    """
    Robust parser for numeric counts or public count-bin strings.
    """
    if pd.isna(x):
        return 1.0

    if isinstance(x, (int, float, np.integer, np.floating)):
        if np.isfinite(x) and x > 0:
            return float(x)
        return 1.0

    s = str(x).strip().lower().replace(",", "")

    if s in {"", "nan", "none", "null", "<na>"}:
        return 1.0

    # If directly numeric string.
    try:
        v = float(s)
        if np.isfinite(v) and v > 0:
            return float(v)
    except Exception:
        pass

    # 100000+, over 100000, etc.
    if "+" in s or "over" in s or "more" in s:
        nums = re.findall(r"\d+", s)
        if nums:
            return float(max(int(n) for n in nums))
        return 100000.0

    nums = [int(n) for n in re.findall(r"\d+", s)]

    if len(nums) >= 2:
        lo, hi = min(nums[0], nums[1]), max(nums[0], nums[1])
        if lo > 0 and hi > 0:
            return float(np.sqrt(lo * hi))

    if len(nums) == 1:
        n = nums[0]
        if n > 0:
            return float(n)

    return 1.0


def parse_weight_series(s: pd.Series) -> pd.Series:
    # Try numeric first.
    numeric = pd.to_numeric(
        s.astype("string").str.replace(",", "", regex=False),
        errors="coerce"
    )

    # If mostly numeric, use numeric.
    if numeric.notna().mean() > 0.8:
        out = numeric.fillna(1.0)
        out = out.where(out > 0, 1.0)
        return out.astype(float)

    return s.apply(representative_count_weight_scalar).astype(float)


def weighted_quantile(values, weights, q):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)

    mask = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[mask]
    weights = weights[mask]

    if len(values) == 0:
        return np.nan

    order = np.argsort(values)
    values = values[order]
    weights = weights[order]

    cdf = np.cumsum(weights)
    cutoff = q * cdf[-1]
    idx = np.searchsorted(cdf, cutoff, side="left")
    idx = min(idx, len(values) - 1)

    return float(values[idx])


def aggregate_one_scope(mapped: pd.DataFrame, scope_name: str) -> pd.DataFrame:
    """
    Aggregate mapped county-pair rows to FAF-pair-year rows.
    """
    if len(mapped) == 0:
        return pd.DataFrame(
            columns=[
                "scope",
                "faf_orig",
                "faf_dest",
                "year",
                "truck_q25",
                "truck_q50",
                "truck_q75",
                "n_fmi_county_pairs",
                "obs_weight_sum",
                "obs_weight_mean",
                "obs_weight_max",
                "input_q25_weighted_mean",
                "input_q50_weighted_mean",
                "input_q75_weighted_mean",
                "input_q50_min",
                "input_q50_max",
                "n_orig_counties",
                "n_dest_counties",
            ]
        )

    rows = []

    grouped = mapped.groupby(["faf_orig", "faf_dest", "year"], sort=False)

    for (faf_orig, faf_dest, year), g in tqdm(
        grouped,
        total=len(grouped),
        desc=f"Aggregating {scope_name}",
    ):
        q25 = g["q25"].to_numpy(dtype=float)
        q50 = g["q50"].to_numpy(dtype=float)
        q75 = g["q75"].to_numpy(dtype=float)
        w = g["obs_weight"].to_numpy(dtype=float)

        # Enforce monotonic row-level quantiles.
        stack = np.vstack([q25, q50, q75])
        q25_fixed = np.nanmin(stack, axis=0)
        q50_fixed = np.nanmedian(stack, axis=0)
        q75_fixed = np.nanmax(stack, axis=0)

        q25, q50, q75 = q25_fixed, q50_fixed, q75_fixed

        lower_gap = np.maximum(q50 - q25, 0)
        upper_gap = np.maximum(q75 - q50, 0)

        q0 = np.maximum(0, q25 - lower_gap)
        q100 = q75 + upper_gap

        # Quantile-matched discrete mixture approximation.
        values = np.concatenate([q0, q25, q50, q75, q100])
        weights = np.concatenate([
            0.125 * w,
            0.250 * w,
            0.250 * w,
            0.250 * w,
            0.125 * w,
        ])

        rows.append({
            "scope": scope_name,
            "faf_orig": str(faf_orig).zfill(3),
            "faf_dest": str(faf_dest).zfill(3),
            "year": int(year),

            "truck_q25": weighted_quantile(values, weights, 0.25),
            "truck_q50": weighted_quantile(values, weights, 0.50),
            "truck_q75": weighted_quantile(values, weights, 0.75),

            "n_fmi_county_pairs": int(len(g)),
            "obs_weight_sum": float(np.nansum(w)),
            "obs_weight_mean": float(np.nanmean(w)),
            "obs_weight_max": float(np.nanmax(w)),

            "input_q25_weighted_mean": float(np.average(q25, weights=w)) if np.nansum(w) > 0 else np.nan,
            "input_q50_weighted_mean": float(np.average(q50, weights=w)) if np.nansum(w) > 0 else np.nan,
            "input_q75_weighted_mean": float(np.average(q75, weights=w)) if np.nansum(w) > 0 else np.nan,

            "input_q50_min": float(np.nanmin(q50)),
            "input_q50_max": float(np.nanmax(q50)),

            "n_orig_counties": int(g["origin_county_fips"].nunique()),
            "n_dest_counties": int(g["destination_county_fips"].nunique()),
        })

    out = pd.DataFrame(rows)
    out = out.sort_values(["year", "faf_orig", "faf_dest"]).reset_index(drop=True)
    return out


# ------------------------------------------------------------
# 2. Clean previous wrong outputs
# ------------------------------------------------------------

if FORCE_REPROCESS:
    print("\nFORCE_REPROCESS=True: previous FMI aggregation outputs will be overwritten.")
    for p in [
        ALL_FINAL_PARQUET,
        ALL_FINAL_CSV,
        EAST_FINAL_PARQUET,
        EAST_FINAL_CSV,
        YEARLY_MANIFEST_PATH,
        SUMMARY_PATH,
        SCHEMA_REPORT_PATH,
    ]:
        if p.exists():
            p.unlink()

    for year in YEARS:
        for p in [
            BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_all.parquet",
            BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_east_plus_gulf.parquet",
            BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_all_preview1000.csv",
            BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_east_plus_gulf_preview1000.csv",
        ]:
            if p.exists():
                p.unlink()

# ------------------------------------------------------------
# 3. Load county_to_faf
# ------------------------------------------------------------

county_to_faf = pd.read_parquet(COUNTY_TO_FAF_PATH).copy()

required_cols = [
    "county_fips",
    "faf_zone",
    "faf_zone_name",
    "east_plus_gulf_county_flag",
]

missing = [c for c in required_cols if c not in county_to_faf.columns]
if missing:
    raise RuntimeError(f"county_to_faf missing columns: {missing}. Columns={list(county_to_faf.columns)}")

county_to_faf["county_fips"] = county_to_faf["county_fips"].astype(str).str.zfill(5)
county_to_faf["faf_zone"] = county_to_faf["faf_zone"].astype(str).str.zfill(3)
county_to_faf["east_plus_gulf_county_flag"] = county_to_faf["east_plus_gulf_county_flag"].astype(bool)

cw = county_to_faf[
    [
        "county_fips",
        "faf_zone",
        "faf_zone_name",
        "east_plus_gulf_county_flag",
    ]
].drop_duplicates("county_fips").copy()

print("\nLoaded county_to_faf:")
print("  rows:", len(cw))
print("  counties:", cw["county_fips"].nunique())
print("  FAF zones:", cw["faf_zone"].nunique())
print("  East-Plus-Gulf counties:", int(cw["east_plus_gulf_county_flag"].sum()))

assert cw["county_fips"].is_unique
assert cw["faf_zone"].nunique() == 132

# ------------------------------------------------------------
# 4. Process FMI by year
# ------------------------------------------------------------

yearly_rows = []
schema_rows = []
all_outputs = []
east_outputs = []

for year in YEARS:
    print("\n" + "=" * 100)
    print("YEAR:", year)
    print("=" * 100)

    fmi_path = find_fmi_file(year)
    schema = detect_fmi_schema(fmi_path, year)
    schema_rows.append(schema)

    print("FMI file:", fmi_path)
    print("Detected schema:")
    for k, v in schema.items():
        if k != "all_columns":
            print(f"  {k}: {v}")

    df_raw = read_fmi_selected(fmi_path, schema)
    print("Raw selected rows:", len(df_raw))
    print("Raw selected columns:", list(df_raw.columns))

    origin_col = schema["origin_col"]
    dest_col = schema["dest_col"]
    q25_col = schema["q25_col"]
    q50_col = schema["q50_col"]
    q75_col = schema["q75_col"]
    weight_col = schema["weight_col"]

    df = pd.DataFrame({
        "origin_county_fips": normalize_fips_series(df_raw[origin_col]),
        "destination_county_fips": normalize_fips_series(df_raw[dest_col]),
        "q25": pd.to_numeric(df_raw[q25_col], errors="coerce"),
        "q50": pd.to_numeric(df_raw[q50_col], errors="coerce"),
        "q75": pd.to_numeric(df_raw[q75_col], errors="coerce"),
        "year": int(year),
    })

    if weight_col is not None:
        df["obs_count_raw"] = df_raw[weight_col]
        df["obs_weight"] = parse_weight_series(df_raw[weight_col])
    else:
        df["obs_count_raw"] = pd.NA
        df["obs_weight"] = 1.0

    del df_raw
    gc.collect()

    before = len(df)

    df = df[
        df["origin_county_fips"].notna()
        & df["destination_county_fips"].notna()
        & df["q25"].notna()
        & df["q50"].notna()
        & df["q75"].notna()
        & (df["q25"] >= 0)
        & (df["q50"] >= 0)
        & (df["q75"] >= 0)
        & df["obs_weight"].notna()
        & (df["obs_weight"] > 0)
    ].copy()

    after = len(df)

    print("Rows after basic clean:", after, "dropped:", before - after)
    print("Origin FIPS sample:", df["origin_county_fips"].head().tolist())
    print("Destination FIPS sample:", df["destination_county_fips"].head().tolist())
    print("Weight summary:")
    print(df["obs_weight"].describe())

    # Join origin county -> FAF.
    df = df.merge(
        cw.rename(
            columns={
                "county_fips": "origin_county_fips",
                "faf_zone": "faf_orig",
                "faf_zone_name": "faf_orig_name",
                "east_plus_gulf_county_flag": "origin_east_plus_gulf_flag",
            }
        ),
        on="origin_county_fips",
        how="left",
    )

    # Join destination county -> FAF.
    df = df.merge(
        cw.rename(
            columns={
                "county_fips": "destination_county_fips",
                "faf_zone": "faf_dest",
                "faf_zone_name": "faf_dest_name",
                "east_plus_gulf_county_flag": "dest_east_plus_gulf_flag",
            }
        ),
        on="destination_county_fips",
        how="left",
    )

    n_unmapped_origin = int(df["faf_orig"].isna().sum())
    n_unmapped_dest = int(df["faf_dest"].isna().sum())

    mapped = df[
        df["faf_orig"].notna()
        & df["faf_dest"].notna()
    ].copy()

    mapped["faf_orig"] = mapped["faf_orig"].astype(str).str.zfill(3)
    mapped["faf_dest"] = mapped["faf_dest"].astype(str).str.zfill(3)

    mapped["both_counties_east_plus_gulf"] = (
        mapped["origin_east_plus_gulf_flag"].fillna(False).astype(bool)
        & mapped["dest_east_plus_gulf_flag"].fillna(False).astype(bool)
    )

    print("Mapped rows:", len(mapped))
    print("Unmapped origin rows:", n_unmapped_origin)
    print("Unmapped dest rows:", n_unmapped_dest)
    print("East-Plus-Gulf mapped rows:", int(mapped["both_counties_east_plus_gulf"].sum()))
    print("Unique all FAF OD pairs:", mapped[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
    print("Unique East FAF OD pairs:", mapped.loc[mapped["both_counties_east_plus_gulf"], ["faf_orig", "faf_dest"]].drop_duplicates().shape[0])

    if SAVE_MAPPED_COUNTY_PAIR_TABLES:
        mapped_out = MAPPED_PAIR_DIR / f"fmi_county_pair_to_faf_pair_{year}.parquet"
        mapped.to_parquet(mapped_out, index=False)
        print("Saved mapped county-pair table:", mapped_out)

    # Aggregate all mapped county pairs.
    all_agg = aggregate_one_scope(mapped, scope_name="all_valid_faf")

    # Aggregate East-Plus-Gulf only.
    east_mapped = mapped[mapped["both_counties_east_plus_gulf"]].copy()
    east_agg = aggregate_one_scope(east_mapped, scope_name="east_plus_gulf")

    # Save per-year.
    all_year_out = BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_all.parquet"
    east_year_out = BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_east_plus_gulf.parquet"

    all_agg.to_parquet(all_year_out, index=False)
    east_agg.to_parquet(east_year_out, index=False)

    all_agg.head(1000).to_csv(
        BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_all_preview1000.csv",
        index=False,
        encoding="utf-8-sig",
    )

    east_agg.head(1000).to_csv(
        BY_YEAR_DIR / f"fmi_faf_pair_quantiles_{year}_east_plus_gulf_preview1000.csv",
        index=False,
        encoding="utf-8-sig",
    )

    all_outputs.append(all_agg)
    east_outputs.append(east_agg)

    yearly_rows.append({
        "year": year,
        "status": "processed_fixed",
        "fmi_path": str(fmi_path),
        "raw_selected_rows": int(before),
        "rows_after_basic_clean": int(after),
        "mapped_rows": int(len(mapped)),
        "unmapped_origin_rows": n_unmapped_origin,
        "unmapped_dest_rows": n_unmapped_dest,
        "east_plus_gulf_mapped_rows": int(mapped["both_counties_east_plus_gulf"].sum()),
        "n_all_faf_pairs": int(len(all_agg)),
        "n_east_faf_pairs": int(len(east_agg)),
        "origin_col": origin_col,
        "dest_col": dest_col,
        "q25_col": q25_col,
        "q50_col": q50_col,
        "q75_col": q75_col,
        "weight_col": weight_col,
        "all_output_path": str(all_year_out),
        "east_output_path": str(east_year_out),
    })

    del df, mapped, east_mapped, all_agg, east_agg
    gc.collect()

# ------------------------------------------------------------
# 5. Combine all years
# ------------------------------------------------------------

all_final = pd.concat(all_outputs, ignore_index=True) if all_outputs else pd.DataFrame()
east_final = pd.concat(east_outputs, ignore_index=True) if east_outputs else pd.DataFrame()

all_final = all_final.sort_values(["year", "faf_orig", "faf_dest"]).reset_index(drop=True)
east_final = east_final.sort_values(["year", "faf_orig", "faf_dest"]).reset_index(drop=True)

all_final.to_parquet(ALL_FINAL_PARQUET, index=False)
all_final.to_csv(ALL_FINAL_CSV, index=False, encoding="utf-8-sig")

east_final.to_parquet(EAST_FINAL_PARQUET, index=False)
east_final.to_csv(EAST_FINAL_CSV, index=False, encoding="utf-8-sig")

yearly_manifest = pd.DataFrame(yearly_rows)
schema_report = pd.DataFrame(schema_rows)

yearly_manifest.to_csv(YEARLY_MANIFEST_PATH, index=False, encoding="utf-8-sig")
schema_report.to_csv(SCHEMA_REPORT_PATH, index=False, encoding="utf-8-sig")

summary = {
    "status": "ok_fixed",
    "years": YEARS,
    "county_to_faf_path": str(COUNTY_TO_FAF_PATH),
    "output_root": str(OUT_ROOT),
    "all_final_parquet": str(ALL_FINAL_PARQUET),
    "all_final_csv": str(ALL_FINAL_CSV),
    "east_final_parquet": str(EAST_FINAL_PARQUET),
    "east_final_csv": str(EAST_FINAL_CSV),
    "yearly_manifest_path": str(YEARLY_MANIFEST_PATH),
    "schema_report_path": str(SCHEMA_REPORT_PATH),
    "n_all_rows": int(len(all_final)),
    "n_east_rows": int(len(east_final)),
    "n_all_unique_years": int(all_final["year"].nunique()) if len(all_final) else 0,
    "n_east_unique_years": int(east_final["year"].nunique()) if len(east_final) else 0,
    "n_all_unique_faf_od_pairs": int(all_final[["faf_orig", "faf_dest"]].drop_duplicates().shape[0]) if len(all_final) else 0,
    "n_east_unique_faf_od_pairs": int(east_final[["faf_orig", "faf_dest"]].drop_duplicates().shape[0]) if len(east_final) else 0,
    "aggregation_method": "quantile_matched_discrete_mixture_from_public_p25_p50_p75",
    "weight_method": "Crossings_or_Movements_numeric_weight_or_count_bin_parser",
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 100)
print("FMI -> FAF AGGREGATION FIXED COMPLETE")
print("=" * 100)
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nYearly manifest:")
display(yearly_manifest)

print("\nAll valid FAF-pair preview:")
display(all_final.head(20))

print("\nEast-Plus-Gulf FAF-pair preview:")
display(east_final.head(20))

# Final assertions.
assert len(all_final) > 0, "All final aggregation is still empty."
assert len(east_final) > 0, "East-Plus-Gulf aggregation is still empty."
assert all_final["year"].nunique() == 7, "All final should cover 7 years."
assert east_final["year"].nunique() == 7, "East final should cover 7 years."
assert (all_final["truck_q25"] <= all_final["truck_q50"]).all()
assert (all_final["truck_q50"] <= all_final["truck_q75"]).all()
assert (east_final["truck_q25"] <= east_final["truck_q50"]).all()
assert (east_final["truck_q50"] <= east_final["truck_q75"]).all()

print("\nOK: FMI -> FAF-pair truck quantile labels are ready.")

FMI county-pair -> FAF-pair aggregation FIXED
FMI_ROOT: E:\NetworkOptimization\pythonProject1\Data\02_fmi
COUNTY_TO_FAF_PATH: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\county_to_faf.parquet
OUT_ROOT: E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated

FORCE_REPROCESS=True: previous FMI aggregation outputs will be overwritten.

Loaded county_to_faf:
  rows: 3144
  counties: 3144
  FAF zones: 132
  East-Plus-Gulf counties: 2510

YEAR: 2018
FMI file: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.parquet
Detected schema:
  year: 2018
  path: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2018\bts_fmi_county_travel_times_2018_xx4g-5dg2.parquet
  origin_col: Origin CTFIPS
  dest_col: Destination CTFIPS
  q25_col: 25th Percentile (minutes)
  q50_col: 50th Percentile (minutes)
  q75_col: 75th Percentile (minutes)
  weight_col: Crossings
Raw selected rows: 2080775
Raw selected columns: ['

Aggregating all_valid_faf:   0%|          | 0/15021 [00:00<?, ?it/s]

Aggregating east_plus_gulf:   0%|          | 0/9936 [00:00<?, ?it/s]


YEAR: 2019
FMI file: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.parquet
Detected schema:
  year: 2019
  path: E:\NetworkOptimization\pythonProject1\Data\02_fmi\2019\bts_fmi_county_travel_times_2019_sn4k-eiea.parquet
  origin_col: Origin CTFIPS
  dest_col: Destination CTFIPS
  q25_col: 25th Percentile (minutes)
  q50_col: 50th Percentile (minutes)
  q75_col: 75th Percentile (minutes)
  weight_col: Movements
Raw selected rows: 4130615
Raw selected columns: ['Origin CTFIPS', 'Destination CTFIPS', '25th Percentile (minutes)', '50th Percentile (minutes)', '75th Percentile (minutes)', 'Movements']
Rows after basic clean: 4130615 dropped: 0
Origin FIPS sample: ['13020', '13020', '13020', '13020', '13020']
Destination FIPS sample: ['13020', '13020', '13110', '13110', '13120']
Weight summary:
count    4.130615e+06
mean     4.439091e+00
std      8.441067e+00
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      3.162278

Aggregating all_valid_faf:   0%|          | 0/16346 [00:00<?, ?it/s]

...


All valid FAF-pair preview:


,scope,faf_orig,faf_dest,year,truck_q25,truck_q50,truck_q75,n_fmi_county_pairs,obs_weight_sum,obs_weight_mean,obs_weight_max,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties
0,all_valid_faf,011,011,2018,1.00,37.02,71.28,110,1541.730999,14.015736,31.622777,33.156860,47.766945,155.222758,1.00,562.50,11,11
1,all_valid_faf,011,012,2018,248.18,943.27,1925.53,22,56.596443,2.572566,3.162278,516.764447,966.672508,2184.157713,173.83,1979.45,11,2
2,all_valid_faf,011,019,2018,48.00,100.10,218.95,558,3706.180871,6.641901,31.622777,95.525005,178.619190,492.856903,1.00,1646.70,11,54
3,all_valid_faf,011,041,2018,1929.44,2498.82,3835.27,16,28.973666,1.810854,3.162278,2088.158470,2701.561659,4109.080662,2058.85,4197.91,8,2
4,all_valid_faf,011,042,2018,1782.55,2332.43,3435.48,8,14.486833,1.810854,3.162278,1980.692788,2536.740261,3894.585715,1972.80,4136.00,8,1
5,all_valid_faf,011,049,2018,1860.02,2576.80,3989.00,60,116.219219,1.936987,3.162278,1993.514928,2623.732939,4062.570833,1743.69,4108.96,8,8
6,all_valid_faf,011,050,2018,404.01,1010.39,1818.68,433,1365.823515,3.154327,31.622777,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63
7,all_valid_faf,011,061,2018,2414.93,2921.98,3998.47,32,62.271887,1.945996,3.162278,2420.813245,2977.466031,4206.878474,2262.08,4093.48,8,5
8,all_valid_faf,011,062,2018,3039.21,4153.81,5892.74,16,16.000000,1.000000,1.000000,3398.950625,4425.448750,6813.383125,3125.40,5892.74,4,4
9,all_valid_faf,011,063,2018,2439.78,3273.48,4479.53,3,3.000000,1.000000,1.000000,2446.170000,3308.870000,4487.110000,3254.07,3399.06,3,1



East-Plus-Gulf FAF-pair preview:


,scope,faf_orig,faf_dest,year,truck_q25,truck_q50,truck_q75,n_fmi_county_pairs,obs_weight_sum,obs_weight_mean,obs_weight_max,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties
0,east_plus_gulf,011,011,2018,1.00,37.02,71.28,110,1541.730999,14.015736,31.622777,33.156860,47.766945,155.222758,1.00,562.50,11,11
1,east_plus_gulf,011,012,2018,248.18,943.27,1925.53,22,56.596443,2.572566,3.162278,516.764447,966.672508,2184.157713,173.83,1979.45,11,2
2,east_plus_gulf,011,019,2018,48.00,100.10,218.95,558,3706.180871,6.641901,31.622777,95.525005,178.619190,492.856903,1.00,1646.70,11,54
3,east_plus_gulf,011,050,2018,404.01,1010.39,1818.68,433,1365.823515,3.154327,31.622777,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63
4,east_plus_gulf,011,091,2018,2579.38,3718.82,5127.33,15,15.000000,1.000000,1.000000,2510.728667,3559.798000,5716.874000,2557.95,4924.82,8,2
5,east_plus_gulf,011,092,2018,2242.10,3084.89,4836.23,28,32.324555,1.154448,3.162278,2252.316883,3218.952240,5135.916817,2362.98,5337.73,8,4
6,east_plus_gulf,011,099,2018,2307.02,3061.08,4858.83,7,7.000000,1.000000,1.000000,2364.197143,3353.897143,5706.734286,2475.53,4970.50,4,2
7,east_plus_gulf,011,101,2018,1731.31,2472.42,3873.20,10,14.324555,1.432456,3.162278,1823.161674,2573.021773,3973.259477,2137.94,3250.61,8,2
8,east_plus_gulf,011,121,2018,899.55,1365.26,2482.71,60,124.868330,2.081139,3.162278,856.048234,1411.309250,2538.148244,480.71,2541.90,11,6
9,east_plus_gulf,011,122,2018,1306.35,1584.57,2828.86,68,91.785054,1.349780,3.162278,1314.750359,1629.469826,2767.383169,1309.43,2242.65,10,7



OK: FMI -> FAF-pair truck quantile labels are ready.


In [4]:
from pathlib import Path
import json
import pandas as pd

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
OUT_ROOT = DATA_ROOT / "08_processed" / "fmi_faf_aggregated"

paths = {
    "all_final": OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_all.parquet",
    "east_final": OUT_ROOT / "fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.parquet",
    "yearly_manifest": OUT_ROOT / "_metadata" / "fmi_faf_aggregation_yearly_manifest_fixed.csv",
    "summary": OUT_ROOT / "_metadata" / "fmi_faf_aggregation_summary_fixed.json",
    "schema": OUT_ROOT / "_metadata" / "fmi_detected_schema_by_year_fixed.csv",
}

for name, p in paths.items():
    print(name, p.exists(), p, round(p.stat().st_size / 1024**2, 4) if p.exists() else None)

all_df = pd.read_parquet(paths["all_final"])
east_df = pd.read_parquet(paths["east_final"])
yearly_manifest = pd.read_csv(paths["yearly_manifest"])
schema = pd.read_csv(paths["schema"])
summary = json.loads(paths["summary"].read_text(encoding="utf-8"))

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nSchema detection:")
display(schema[[
    "year",
    "origin_col",
    "dest_col",
    "q25_col",
    "q50_col",
    "q75_col",
    "weight_col",
]])

print("\nYearly manifest:")
display(yearly_manifest)

print("\nAll valid FAF-pair labels:")
print("rows:", len(all_df))
print("years:", sorted(all_df["year"].unique().tolist()))
print("unique FAF OD pairs:", all_df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("q monotonic check:", bool((all_df["truck_q25"] <= all_df["truck_q50"]).all() and (all_df["truck_q50"] <= all_df["truck_q75"]).all()))

print("\nEast-Plus-Gulf FAF-pair labels:")
print("rows:", len(east_df))
print("years:", sorted(east_df["year"].unique().tolist()))
print("unique FAF OD pairs:", east_df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("q monotonic check:", bool((east_df["truck_q25"] <= east_df["truck_q50"]).all() and (east_df["truck_q50"] <= east_df["truck_q75"]).all()))

print("\nCoverage by year, all:")
display(
    all_df.groupby("year")
    .agg(
        n_faf_pairs=("faf_orig", "size"),
        total_county_pairs=("n_fmi_county_pairs", "sum"),
        median_county_pairs_per_faf_pair=("n_fmi_county_pairs", "median"),
        median_q50=("truck_q50", "median"),
    )
    .reset_index()
)

print("\nCoverage by year, East-Plus-Gulf:")
display(
    east_df.groupby("year")
    .agg(
        n_faf_pairs=("faf_orig", "size"),
        total_county_pairs=("n_fmi_county_pairs", "sum"),
        median_county_pairs_per_faf_pair=("n_fmi_county_pairs", "median"),
        median_q50=("truck_q50", "median"),
    )
    .reset_index()
)

display(all_df.head(20))
display(east_df.head(20))

assert len(all_df) > 0
assert len(east_df) > 0
assert all_df["year"].nunique() == 7
assert east_df["year"].nunique() == 7
assert (all_df["truck_q25"] <= all_df["truck_q50"]).all()
assert (all_df["truck_q50"] <= all_df["truck_q75"]).all()
assert (east_df["truck_q25"] <= east_df["truck_q50"]).all()
assert (east_df["truck_q50"] <= east_df["truck_q75"]).all()

print("\nOK: fixed FMI -> FAF-pair truck quantile labels are ready.")

all_final True E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\fmi_faf_pair_quantiles_2018_2024_all.parquet 8.1494
east_final True E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.parquet 5.4273
yearly_manifest True E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\_metadata\fmi_faf_aggregation_yearly_manifest_fixed.csv 0.004
summary True E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\_metadata\fmi_faf_aggregation_summary_fixed.json 0.0016
schema True E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\_metadata\fmi_detected_schema_by_year_fixed.csv 0.0029

Summary:
{
  "status": "ok_fixed",
  "years": [
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ],
  "county_to_faf_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\county_to_faf.parquet",
  "output_r

,year,origin_col,dest_col,q25_col,q50_col,q75_col,weight_col
0,2018,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Crossings
1,2019,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements
2,2020,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements
3,2021,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements
4,2022,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements
5,2023,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements
6,2024,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements



Yearly manifest:


,year,status,fmi_path,raw_selected_rows,rows_after_basic_clean,mapped_rows,unmapped_origin_rows,unmapped_dest_rows,east_plus_gulf_mapped_rows,n_all_faf_pairs,n_east_faf_pairs,origin_col,dest_col,q25_col,q50_col,q75_col,weight_col,all_output_path,east_output_path
0,2018,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,2080775,2080775,2074924,3096,3069,1590164,15021,9936,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Crossings,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
1,2019,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,4130615,4130615,4069609,30964,31625,3062398,16346,10704,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
2,2020,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,4429137,4429137,4381402,0,47735,3274940,16458,10761,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
3,2021,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,4248604,4248604,4198449,0,50155,3150679,16389,10721,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
4,2022,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,3807134,3807134,3761413,0,45721,2849256,16238,10651,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
5,2023,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,3642150,3642150,3642150,0,0,2764474,16157,10625,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...
6,2024,processed_fixed,E:\NetworkOptimization\pythonProject1\Data\02_...,3585730,3585730,3413986,2046,171744,2620861,16066,10574,Origin CTFIPS,Destination CTFIPS,25th Percentile (minutes),50th Percentile (minutes),75th Percentile (minutes),Movements,E:\NetworkOptimization\pythonProject1\Data\08_...,E:\NetworkOptimization\pythonProject1\Data\08_...



All valid FAF-pair labels:
rows: 112675
years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
unique FAF OD pairs: 16469
q monotonic check: True

East-Plus-Gulf FAF-pair labels:
rows: 73972
years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
unique FAF OD pairs: 10762
q monotonic check: True

Coverage by year, all:


...


Coverage by year, East-Plus-Gulf:


,year,n_faf_pairs,total_county_pairs,median_county_pairs_per_faf_pair,median_q50
0,2018,9936,1590164,54.0,2197.420
1,2019,10704,3062398,85.0,2425.840
2,2020,10761,3274940,90.0,2324.670
3,2021,10721,3150679,89.0,2331.850
4,2022,10651,2849256,82.0,2289.320
5,2023,10625,2764474,80.0,2322.230
6,2024,10574,2620861,77.0,2496.035


,scope,faf_orig,faf_dest,year,truck_q25,truck_q50,truck_q75,n_fmi_county_pairs,obs_weight_sum,obs_weight_mean,obs_weight_max,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties
0,all_valid_faf,011,011,2018,1.00,37.02,71.28,110,1541.730999,14.015736,31.622777,33.156860,47.766945,155.222758,1.00,562.50,11,11
1,all_valid_faf,011,012,2018,248.18,943.27,1925.53,22,56.596443,2.572566,3.162278,516.764447,966.672508,2184.157713,173.83,1979.45,11,2
2,all_valid_faf,011,019,2018,48.00,100.10,218.95,558,3706.180871,6.641901,31.622777,95.525005,178.619190,492.856903,1.00,1646.70,11,54
3,all_valid_faf,011,041,2018,1929.44,2498.82,3835.27,16,28.973666,1.810854,3.162278,2088.158470,2701.561659,4109.080662,2058.85,4197.91,8,2
4,all_valid_faf,011,042,2018,1782.55,2332.43,3435.48,8,14.486833,1.810854,3.162278,1980.692788,2536.740261,3894.585715,1972.80,4136.00,8,1
5,all_valid_faf,011,049,2018,1860.02,2576.80,3989.00,60,116.219219,1.936987,3.162278,1993.514928,2623.732939,4062.570833,1743.69,4108.96,8,8
6,all_valid_faf,011,050,2018,404.01,1010.39,1818.68,433,1365.823515,3.154327,31.622777,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63
7,all_valid_faf,011,061,2018,2414.93,2921.98,3998.47,32,62.271887,1.945996,3.162278,2420.813245,2977.466031,4206.878474,2262.08,4093.48,8,5
8,all_valid_faf,011,062,2018,3039.21,4153.81,5892.74,16,16.000000,1.000000,1.000000,3398.950625,4425.448750,6813.383125,3125.40,5892.74,4,4
9,all_valid_faf,011,063,2018,2439.78,3273.48,4479.53,3,3.000000,1.000000,1.000000,2446.170000,3308.870000,4487.110000,3254.07,3399.06,3,1


,scope,faf_orig,faf_dest,year,truck_q25,truck_q50,truck_q75,n_fmi_county_pairs,obs_weight_sum,obs_weight_mean,obs_weight_max,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties
0,east_plus_gulf,011,011,2018,1.00,37.02,71.28,110,1541.730999,14.015736,31.622777,33.156860,47.766945,155.222758,1.00,562.50,11,11
1,east_plus_gulf,011,012,2018,248.18,943.27,1925.53,22,56.596443,2.572566,3.162278,516.764447,966.672508,2184.157713,173.83,1979.45,11,2
2,east_plus_gulf,011,019,2018,48.00,100.10,218.95,558,3706.180871,6.641901,31.622777,95.525005,178.619190,492.856903,1.00,1646.70,11,54
3,east_plus_gulf,011,050,2018,404.01,1010.39,1818.68,433,1365.823515,3.154327,31.622777,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63
4,east_plus_gulf,011,091,2018,2579.38,3718.82,5127.33,15,15.000000,1.000000,1.000000,2510.728667,3559.798000,5716.874000,2557.95,4924.82,8,2
5,east_plus_gulf,011,092,2018,2242.10,3084.89,4836.23,28,32.324555,1.154448,3.162278,2252.316883,3218.952240,5135.916817,2362.98,5337.73,8,4
6,east_plus_gulf,011,099,2018,2307.02,3061.08,4858.83,7,7.000000,1.000000,1.000000,2364.197143,3353.897143,5706.734286,2475.53,4970.50,4,2
7,east_plus_gulf,011,101,2018,1731.31,2472.42,3873.20,10,14.324555,1.432456,3.162278,1823.161674,2573.021773,3973.259477,2137.94,3250.61,8,2
8,east_plus_gulf,011,121,2018,899.55,1365.26,2482.71,60,124.868330,2.081139,3.162278,856.048234,1411.309250,2538.148244,480.71,2541.90,11,6
9,east_plus_gulf,011,122,2018,1306.35,1584.57,2828.86,68,91.785054,1.349780,3.162278,1314.750359,1629.469826,2767.383169,1309.43,2242.65,10,7



OK: fixed FMI -> FAF-pair truck quantile labels are ready.


In [5]:
# ============================================================
# FREIGHT-MNet Processing Step 5:
# Process FAF5.7.1 OD demand for modes 1 / 2 / 5, years 2018–2024
#
# Input:
#   E:\NetworkOptimization\pythonProject1\Data\01_faf\flows_regional\FAF5.7.1.csv
#
# Outputs:
#   Data\08_processed\graph_inputs\
#     - faf_od_demand_modes_1_2_5_2018_2024_long.parquet
#     - faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_long.parquet
#     - faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet
#     - faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet
#
# Notes:
#   - Long table keeps commodity-level SCTG2 rows.
#   - OD-mode-year table aggregates across commodities.
#   - East-Plus-Gulf table keeps OD pairs whose origin and destination FAF zones
#     both have East-Plus-Gulf counties.
# ============================================================

from pathlib import Path
import json
import gc
import re

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")

FAF_PATH = DATA_ROOT / "01_faf" / "flows_regional" / "FAF5.7.1.csv"

COUNTY_TO_FAF_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"
EAST_FAF_ZONES_PATH = COUNTY_TO_FAF_DIR / "east_plus_gulf_faf_zones.parquet"

OUT_DIR = DATA_ROOT / "08_processed" / "graph_inputs"
META_DIR = OUT_DIR / "_metadata"

OUT_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2018, 2025))
KEEP_MODES = {1, 2, 5}

MODE_NAMES = {
    1: "truck",
    2: "rail",
    5: "multiple_modes_mail_intermodal_like",
}

LONG_ALL_PARQUET = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_long.parquet"
LONG_EAST_PARQUET = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_long.parquet"

AGG_ALL_PARQUET = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet"
AGG_ALL_CSV = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.csv"

AGG_EAST_PARQUET = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet"
AGG_EAST_CSV = OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.csv"

SUMMARY_PATH = META_DIR / "faf_od_demand_processing_summary.json"
SCHEMA_PATH = META_DIR / "faf_od_demand_detected_schema.json"

CHUNKSIZE = 250_000
FORCE_REPROCESS = True

print("=" * 100)
print("FAF OD demand processing: modes 1 / 2 / 5, years 2018–2024")
print("FAF_PATH:", FAF_PATH)
print("EAST_FAF_ZONES_PATH:", EAST_FAF_ZONES_PATH)
print("OUT_DIR:", OUT_DIR)
print("=" * 100)

if not FAF_PATH.exists():
    raise FileNotFoundError(f"Missing FAF file: {FAF_PATH}")

if not EAST_FAF_ZONES_PATH.exists():
    raise FileNotFoundError(f"Missing East-Plus-Gulf FAF zone summary: {EAST_FAF_ZONES_PATH}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def normalize_colname(c: str) -> str:
    return "".join(ch.lower() for ch in str(c) if ch.isalnum())


def find_col(columns, candidates, required=True, label="column"):
    norm_map = {normalize_colname(c): c for c in columns}

    for cand in candidates:
        key = normalize_colname(cand)
        if key in norm_map:
            return norm_map[key]

    if required:
        raise RuntimeError(
            f"Could not detect {label}. Candidates={candidates}. Available={list(columns)}"
        )

    return None


def find_metric_col(columns, metric: str, year: int) -> str:
    """
    Find columns like:
      tons_2018, value_2018, tmiles_2018
    robustly.
    """
    candidates = [
        f"{metric}_{year}",
        f"{metric}{year}",
        f"{metric.upper()}_{year}",
        f"{metric.upper()}{year}",
    ]

    return find_col(
        columns,
        candidates,
        required=True,
        label=f"{metric}_{year}",
    )


def normalize_faf_zone_series(s: pd.Series) -> pd.Series:
    out = s.astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.str.replace(r"[^0-9]", "", regex=True)
    out = out.where(out.str.len() > 0, pd.NA)
    return out.str.zfill(3)


def clean_numeric_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float)


def remove_if_exists(path: Path):
    if path.exists():
        path.unlink()


# ------------------------------------------------------------
# 2. Detect FAF schema
# ------------------------------------------------------------

header = pd.read_csv(FAF_PATH, nrows=0)
columns = list(header.columns)

dms_orig_col = find_col(columns, ["dms_orig", "DMS_ORIG"], label="dms_orig")
dms_dest_col = find_col(columns, ["dms_dest", "DMS_DEST"], label="dms_dest")
dms_mode_col = find_col(columns, ["dms_mode", "DMS_MODE"], label="dms_mode")
sctg2_col = find_col(columns, ["sctg2", "SCTG2"], label="sctg2")

# Optional fields.
fr_orig_col = find_col(columns, ["fr_orig", "FR_ORIG"], required=False, label="fr_orig")
fr_dest_col = find_col(columns, ["fr_dest", "FR_DEST"], required=False, label="fr_dest")

metric_cols = {}
for year in YEARS:
    metric_cols[year] = {
        "tons": find_metric_col(columns, "tons", year),
        "value": find_metric_col(columns, "value", year),
        "tmiles": find_metric_col(columns, "tmiles", year),
    }

schema = {
    "dms_orig_col": dms_orig_col,
    "dms_dest_col": dms_dest_col,
    "dms_mode_col": dms_mode_col,
    "sctg2_col": sctg2_col,
    "fr_orig_col": fr_orig_col,
    "fr_dest_col": fr_dest_col,
    "metric_cols": metric_cols,
    "all_columns": columns,
}

SCHEMA_PATH.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding="utf-8")

print("\nDetected schema:")
print(json.dumps({k: v for k, v in schema.items() if k != "all_columns"}, indent=2, ensure_ascii=False))

# Columns to read.
usecols = [
    dms_orig_col,
    dms_dest_col,
    dms_mode_col,
    sctg2_col,
]

for optional in [fr_orig_col, fr_dest_col]:
    if optional is not None:
        usecols.append(optional)

for year in YEARS:
    usecols.extend(metric_cols[year].values())

usecols = list(dict.fromkeys(usecols))

# ------------------------------------------------------------
# 3. Load East-Plus-Gulf FAF zone set
# ------------------------------------------------------------

east_faf = pd.read_parquet(EAST_FAF_ZONES_PATH).copy()

required_east_cols = ["faf_zone", "east_plus_gulf_faf_flag_any_county"]
missing = [c for c in required_east_cols if c not in east_faf.columns]
if missing:
    raise RuntimeError(f"east_plus_gulf_faf_zones missing columns: {missing}. Columns={list(east_faf.columns)}")

east_faf["faf_zone"] = east_faf["faf_zone"].astype(str).str.zfill(3)
east_zone_set = set(
    east_faf.loc[east_faf["east_plus_gulf_faf_flag_any_county"].astype(bool), "faf_zone"]
)

valid_zone_set = set(east_faf["faf_zone"].astype(str))

print("\nFAF zone sets:")
print("  valid FAF zones:", len(valid_zone_set))
print("  East-Plus-Gulf FAF zones:", len(east_zone_set))

# ------------------------------------------------------------
# 4. Clean previous outputs
# ------------------------------------------------------------

if FORCE_REPROCESS:
    for p in [
        LONG_ALL_PARQUET,
        LONG_EAST_PARQUET,
        AGG_ALL_PARQUET,
        AGG_ALL_CSV,
        AGG_EAST_PARQUET,
        AGG_EAST_CSV,
        SUMMARY_PATH,
    ]:
        remove_if_exists(p)

# ------------------------------------------------------------
# 5. Stream process FAF CSV
# ------------------------------------------------------------

writer_all = None
writer_east = None

agg_all_parts = []
agg_east_parts = []

total_raw_rows = 0
total_mode_rows = 0
total_valid_zone_rows = 0
total_long_rows = 0
total_east_long_rows = 0
chunks_processed = 0

reader = pd.read_csv(
    FAF_PATH,
    usecols=usecols,
    chunksize=CHUNKSIZE,
    low_memory=False,
)

for chunk in tqdm(reader, desc="Processing FAF chunks"):
    chunks_processed += 1
    total_raw_rows += len(chunk)

    # Standardize key fields.
    chunk["faf_orig"] = normalize_faf_zone_series(chunk[dms_orig_col])
    chunk["faf_dest"] = normalize_faf_zone_series(chunk[dms_dest_col])

    chunk["dms_mode"] = pd.to_numeric(chunk[dms_mode_col], errors="coerce").astype("Int64")
    chunk["sctg2"] = chunk[sctg2_col].astype("string").str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(2)

    # Filter modes.
    chunk = chunk[chunk["dms_mode"].isin(KEEP_MODES)].copy()
    total_mode_rows += len(chunk)

    # Filter valid domestic FAF zones.
    chunk = chunk[
        chunk["faf_orig"].isin(valid_zone_set)
        & chunk["faf_dest"].isin(valid_zone_set)
    ].copy()

    total_valid_zone_rows += len(chunk)

    if len(chunk) == 0:
        continue

    # Build long rows for each year.
    long_parts = []

    for year in YEARS:
        tons_col = metric_cols[year]["tons"]
        value_col = metric_cols[year]["value"]
        tmiles_col = metric_cols[year]["tmiles"]

        part = pd.DataFrame({
            "faf_orig": chunk["faf_orig"].astype(str),
            "faf_dest": chunk["faf_dest"].astype(str),
            "dms_mode": chunk["dms_mode"].astype(int),
            "mode_name": chunk["dms_mode"].astype(int).map(MODE_NAMES),
            "sctg2": chunk["sctg2"].astype(str),
            "year": int(year),
            "tons": clean_numeric_series(chunk[tons_col]),
            "value": clean_numeric_series(chunk[value_col]),
            "tmiles": clean_numeric_series(chunk[tmiles_col]),
        })

        long_parts.append(part)

    long_df = pd.concat(long_parts, ignore_index=True)

    # Drop all-zero rows.
    long_df = long_df[
        (long_df["tons"] != 0)
        | (long_df["value"] != 0)
        | (long_df["tmiles"] != 0)
    ].copy()

    if len(long_df) == 0:
        continue

    long_df["east_plus_gulf_orig_flag"] = long_df["faf_orig"].isin(east_zone_set)
    long_df["east_plus_gulf_dest_flag"] = long_df["faf_dest"].isin(east_zone_set)
    long_df["east_plus_gulf_faf_pair_flag"] = (
        long_df["east_plus_gulf_orig_flag"]
        & long_df["east_plus_gulf_dest_flag"]
    )

    # Ensure types.
    long_df["dms_mode"] = long_df["dms_mode"].astype("int16")
    long_df["year"] = long_df["year"].astype("int16")
    long_df["tons"] = long_df["tons"].astype("float64")
    long_df["value"] = long_df["value"].astype("float64")
    long_df["tmiles"] = long_df["tmiles"].astype("float64")

    total_long_rows += len(long_df)

    # Write all long table.
    table_all = pa.Table.from_pandas(long_df, preserve_index=False)

    if writer_all is None:
        writer_all = pq.ParquetWriter(LONG_ALL_PARQUET, table_all.schema, compression="zstd")

    writer_all.write_table(table_all)

    # East long table.
    east_long_df = long_df[long_df["east_plus_gulf_faf_pair_flag"]].copy()
    total_east_long_rows += len(east_long_df)

    if len(east_long_df) > 0:
        table_east = pa.Table.from_pandas(east_long_df, preserve_index=False)

        if writer_east is None:
            writer_east = pq.ParquetWriter(LONG_EAST_PARQUET, table_east.schema, compression="zstd")

        writer_east.write_table(table_east)

    # Chunk-level aggregation across commodities.
    group_cols = [
        "faf_orig",
        "faf_dest",
        "dms_mode",
        "mode_name",
        "year",
    ]

    agg_all = (
        long_df
        .groupby(group_cols, as_index=False)
        .agg(
            tons=("tons", "sum"),
            value=("value", "sum"),
            tmiles=("tmiles", "sum"),
            n_sctg2=("sctg2", "nunique"),
            n_records=("sctg2", "size"),
            east_plus_gulf_orig_flag=("east_plus_gulf_orig_flag", "max"),
            east_plus_gulf_dest_flag=("east_plus_gulf_dest_flag", "max"),
            east_plus_gulf_faf_pair_flag=("east_plus_gulf_faf_pair_flag", "max"),
        )
    )

    agg_all_parts.append(agg_all)

    if len(east_long_df) > 0:
        agg_east = (
            east_long_df
            .groupby(group_cols, as_index=False)
            .agg(
                tons=("tons", "sum"),
                value=("value", "sum"),
                tmiles=("tmiles", "sum"),
                n_sctg2=("sctg2", "nunique"),
                n_records=("sctg2", "size"),
                east_plus_gulf_orig_flag=("east_plus_gulf_orig_flag", "max"),
                east_plus_gulf_dest_flag=("east_plus_gulf_dest_flag", "max"),
                east_plus_gulf_faf_pair_flag=("east_plus_gulf_faf_pair_flag", "max"),
            )
        )

        agg_east_parts.append(agg_east)

    del chunk, long_df, long_parts, table_all
    if "east_long_df" in locals():
        del east_long_df
    gc.collect()

if writer_all is not None:
    writer_all.close()

if writer_east is not None:
    writer_east.close()

# ------------------------------------------------------------
# 6. Final aggregation across chunks
# ------------------------------------------------------------

if agg_all_parts:
    agg_all_final = pd.concat(agg_all_parts, ignore_index=True)
    agg_all_final = (
        agg_all_final
        .groupby(["faf_orig", "faf_dest", "dms_mode", "mode_name", "year"], as_index=False)
        .agg(
            tons=("tons", "sum"),
            value=("value", "sum"),
            tmiles=("tmiles", "sum"),
            n_sctg2=("n_sctg2", "max"),
            n_records=("n_records", "sum"),
            east_plus_gulf_orig_flag=("east_plus_gulf_orig_flag", "max"),
            east_plus_gulf_dest_flag=("east_plus_gulf_dest_flag", "max"),
            east_plus_gulf_faf_pair_flag=("east_plus_gulf_faf_pair_flag", "max"),
        )
        .sort_values(["year", "faf_orig", "faf_dest", "dms_mode"])
        .reset_index(drop=True)
    )
else:
    agg_all_final = pd.DataFrame()

if agg_east_parts:
    agg_east_final = pd.concat(agg_east_parts, ignore_index=True)
    agg_east_final = (
        agg_east_final
        .groupby(["faf_orig", "faf_dest", "dms_mode", "mode_name", "year"], as_index=False)
        .agg(
            tons=("tons", "sum"),
            value=("value", "sum"),
            tmiles=("tmiles", "sum"),
            n_sctg2=("n_sctg2", "max"),
            n_records=("n_records", "sum"),
            east_plus_gulf_orig_flag=("east_plus_gulf_orig_flag", "max"),
            east_plus_gulf_dest_flag=("east_plus_gulf_dest_flag", "max"),
            east_plus_gulf_faf_pair_flag=("east_plus_gulf_faf_pair_flag", "max"),
        )
        .sort_values(["year", "faf_orig", "faf_dest", "dms_mode"])
        .reset_index(drop=True)
    )
else:
    agg_east_final = pd.DataFrame()

# Save aggregate outputs.
agg_all_final.to_parquet(AGG_ALL_PARQUET, index=False)
agg_all_final.to_csv(AGG_ALL_CSV, index=False, encoding="utf-8-sig")

agg_east_final.to_parquet(AGG_EAST_PARQUET, index=False)
agg_east_final.to_csv(AGG_EAST_CSV, index=False, encoding="utf-8-sig")

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

summary = {
    "status": "ok",
    "faf_input_path": str(FAF_PATH),
    "east_faf_zones_path": str(EAST_FAF_ZONES_PATH),
    "years": YEARS,
    "keep_modes": sorted(list(KEEP_MODES)),
    "mode_names": MODE_NAMES,
    "chunks_processed": int(chunks_processed),
    "total_raw_rows": int(total_raw_rows),
    "total_rows_after_mode_filter": int(total_mode_rows),
    "total_rows_after_valid_faf_filter": int(total_valid_zone_rows),
    "total_long_rows_written_all": int(total_long_rows),
    "total_long_rows_written_east_plus_gulf": int(total_east_long_rows),
    "n_agg_all_rows": int(len(agg_all_final)),
    "n_agg_east_rows": int(len(agg_east_final)),
    "n_all_unique_faf_od_pairs": int(agg_all_final[["faf_orig", "faf_dest"]].drop_duplicates().shape[0]) if len(agg_all_final) else 0,
    "n_east_unique_faf_od_pairs": int(agg_east_final[["faf_orig", "faf_dest"]].drop_duplicates().shape[0]) if len(agg_east_final) else 0,
    "long_all_parquet": str(LONG_ALL_PARQUET),
    "long_east_parquet": str(LONG_EAST_PARQUET),
    "agg_all_parquet": str(AGG_ALL_PARQUET),
    "agg_east_parquet": str(AGG_EAST_PARQUET),
    "schema_path": str(SCHEMA_PATH),
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 100)
print("FAF OD DEMAND PROCESSING COMPLETE")
print("=" * 100)
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nAggregate all preview:")
display(agg_all_final.head(20))

print("\nAggregate East-Plus-Gulf preview:")
display(agg_east_final.head(20))

# Final assertions.
assert len(agg_all_final) > 0, "All-scope FAF demand aggregate is empty."
assert len(agg_east_final) > 0, "East-Plus-Gulf FAF demand aggregate is empty."
assert set(agg_all_final["dms_mode"].unique()).issubset(KEEP_MODES)
assert set(agg_east_final["dms_mode"].unique()).issubset(KEEP_MODES)
assert agg_all_final["year"].nunique() == 7
assert agg_east_final["year"].nunique() == 7

print("\nOK: FAF OD demand modes 1/2/5 are ready.")

FAF OD demand processing: modes 1 / 2 / 5, years 2018–2024
FAF_PATH: E:\NetworkOptimization\pythonProject1\Data\01_faf\flows_regional\FAF5.7.1.csv
EAST_FAF_ZONES_PATH: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\east_plus_gulf_faf_zones.parquet
OUT_DIR: E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs

Detected schema:
{
  "dms_orig_col": "dms_orig",
  "dms_dest_col": "dms_dest",
  "dms_mode_col": "dms_mode",
  "sctg2_col": "sctg2",
  "fr_orig_col": "fr_orig",
  "fr_dest_col": "fr_dest",
  "metric_cols": {
    "2018": {
      "tons": "tons_2018",
      "value": "value_2018",
      "tmiles": "tmiles_2018"
    },
    "2019": {
      "tons": "tons_2019",
      "value": "value_2019",
      "tmiles": "tmiles_2019"
    },
    "2020": {
      "tons": "tons_2020",
      "value": "value_2020",
      "tmiles": "tmiles_2020"
    },
    "2021": {
      "tons": "tons_2021",
      "value": "value_2021",
      "tmiles": "tmiles_2021"
    },
    "2022": {

Processing FAF chunks: 0it [00:00, ?it/s]


FAF OD DEMAND PROCESSING COMPLETE
{
  "status": "ok",
  "faf_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\01_faf\\flows_regional\\FAF5.7.1.csv",
  "east_faf_zones_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\east_plus_gulf_faf_zones.parquet",
  "years": [
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
  ],
  "keep_modes": [
    1,
    2,
    5
  ],
  "mode_names": {
    "1": "truck",
    "2": "rail",
    "5": "multiple_modes_mail_intermodal_like"
  },
  "chunks_processed": 11,
  "total_raw_rows": 2671386,
  "total_rows_after_mode_filter": 1817368,
  "total_rows_after_valid_faf_filter": 1817368,
  "total_long_rows_written_all": 8782844,
  "total_long_rows_written_east_plus_gulf": 6039668,
  "n_agg_all_rows": 307713,
  "n_agg_east_rows": 195481,
  "n_all_unique_faf_od_pairs": 17399,
  "n_east_unique_faf_od_pairs": 10807,
  "long_all_parquet": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\graph_

,faf_orig,faf_dest,dms_mode,mode_name,year,tons,value,tmiles,n_sctg2,n_records,east_plus_gulf_orig_flag,east_plus_gulf_dest_flag,east_plus_gulf_faf_pair_flag
0,011,011,1,truck,2018,35880.873479,15326.185825,1483.641332,29,149,True,True,True
1,011,011,2,rail,2018,2027.859777,14785.299615,128.499054,6,7,True,True,True
2,011,011,5,multiple_modes_mail_intermodal_like,2018,120.107599,1430.297501,6.632377,18,26,True,True,True
3,011,012,1,truck,2018,1003.790981,1260.526660,256.687371,20,329,True,True,True
4,011,012,2,rail,2018,224.597307,211.560396,66.574569,10,118,True,True,True
5,011,012,5,multiple_modes_mail_intermodal_like,2018,42.584159,536.779846,11.718068,8,135,True,True,True
6,011,019,1,truck,2018,10307.348476,8077.511538,1113.272652,28,199,True,True,True
7,011,019,2,rail,2018,21.458409,1.282200,2.499999,1,2,True,True,True
8,011,019,5,multiple_modes_mail_intermodal_like,2018,307.220916,2119.505217,36.004755,14,23,True,True,True
9,011,020,5,multiple_modes_mail_intermodal_like,2018,0.037479,0.233189,0.168731,8,10,True,False,False



Aggregate East-Plus-Gulf preview:


,faf_orig,faf_dest,dms_mode,mode_name,year,tons,value,tmiles,n_sctg2,n_records,east_plus_gulf_orig_flag,east_plus_gulf_dest_flag,east_plus_gulf_faf_pair_flag
0,011,011,1,truck,2018,35880.873479,15326.185825,1483.641332,29,149,True,True,True
1,011,011,2,rail,2018,2027.859777,14785.299615,128.499054,6,7,True,True,True
2,011,011,5,multiple_modes_mail_intermodal_like,2018,120.107599,1430.297501,6.632377,18,26,True,True,True
3,011,012,1,truck,2018,1003.790981,1260.526660,256.687371,20,329,True,True,True
4,011,012,2,rail,2018,224.597307,211.560396,66.574569,10,118,True,True,True
5,011,012,5,multiple_modes_mail_intermodal_like,2018,42.584159,536.779846,11.718068,8,135,True,True,True
6,011,019,1,truck,2018,10307.348476,8077.511538,1113.272652,28,199,True,True,True
7,011,019,2,rail,2018,21.458409,1.282200,2.499999,1,2,True,True,True
8,011,019,5,multiple_modes_mail_intermodal_like,2018,307.220916,2119.505217,36.004755,14,23,True,True,True
9,011,050,1,truck,2018,274.187271,236.354833,105.209681,14,24,True,True,True



OK: FAF OD demand modes 1/2/5 are ready.


In [6]:
from pathlib import Path
import json
import pandas as pd

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
OUT_DIR = DATA_ROOT / "08_processed" / "graph_inputs"

paths = {
    "long_all": OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_long.parquet",
    "long_east": OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_long.parquet",
    "agg_all": OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet",
    "agg_east": OUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet",
    "summary": OUT_DIR / "_metadata" / "faf_od_demand_processing_summary.json",
}

for name, p in paths.items():
    print(name, p.exists(), p, round(p.stat().st_size / 1024**2, 4) if p.exists() else None)

agg_all = pd.read_parquet(paths["agg_all"])
agg_east = pd.read_parquet(paths["agg_east"])
summary = json.loads(paths["summary"].read_text(encoding="utf-8"))

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nAll demand aggregate:")
print("rows:", len(agg_all))
print("years:", sorted(agg_all["year"].unique().tolist()))
print("modes:", sorted(agg_all["dms_mode"].unique().tolist()))
print("unique FAF OD pairs:", agg_all[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("total tons:", agg_all["tons"].sum())
print("total value:", agg_all["value"].sum())
print("total tmiles:", agg_all["tmiles"].sum())

print("\nEast-Plus-Gulf demand aggregate:")
print("rows:", len(agg_east))
print("years:", sorted(agg_east["year"].unique().tolist()))
print("modes:", sorted(agg_east["dms_mode"].unique().tolist()))
print("unique FAF OD pairs:", agg_east[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("total tons:", agg_east["tons"].sum())
print("total value:", agg_east["value"].sum())
print("total tmiles:", agg_east["tmiles"].sum())

print("\nDemand by mode, all:")
display(
    agg_all
    .groupby(["year", "dms_mode", "mode_name"])
    .agg(
        n_rows=("faf_orig", "size"),
        tons=("tons", "sum"),
        value=("value", "sum"),
        tmiles=("tmiles", "sum"),
    )
    .reset_index()
)

print("\nDemand by mode, East-Plus-Gulf:")
display(
    agg_east
    .groupby(["year", "dms_mode", "mode_name"])
    .agg(
        n_rows=("faf_orig", "size"),
        tons=("tons", "sum"),
        value=("value", "sum"),
        tmiles=("tmiles", "sum"),
    )
    .reset_index()
)

display(agg_all.head(20))
display(agg_east.head(20))

assert len(agg_all) > 0
assert len(agg_east) > 0
assert agg_all["year"].nunique() == 7
assert agg_east["year"].nunique() == 7
assert set(agg_all["dms_mode"].unique()).issubset({1, 2, 5})
assert set(agg_east["dms_mode"].unique()).issubset({1, 2, 5})

print("\nOK: FAF demand tables are ready.")

long_all True E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_long.parquet 170.0178
long_east True E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_long.parquet 115.3389
agg_all True E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet 8.5416
agg_east True E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet 5.7034
summary True E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\_metadata\faf_od_demand_processing_summary.json 0.0016

Summary:
{
  "status": "ok",
  "faf_input_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\01_faf\\flows_regional\\FAF5.7.1.csv",
  "east_faf_zones_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\09_crosswalks\\county_to_faf\\east_plus_gulf_fa

,year,dms_mode,mode_name,n_rows,tons,value,tmiles
0,2018,1,truck,16713,1.295855e+07,1.401684e+07,2.439712e+06
1,2018,2,rail,10151,1.639486e+06,5.852998e+05,1.104616e+06
2,2018,5,multiple_modes_mail_intermodal_like,17278,6.719989e+05,2.627811e+06,5.841513e+05
3,2019,1,truck,16735,1.285162e+07,1.380869e+07,2.400781e+06
4,2019,2,rail,10091,1.598599e+06,5.836981e+05,1.057931e+06
5,2019,5,multiple_modes_mail_intermodal_like,17266,6.534026e+05,2.582277e+06,5.622588e+05
6,2020,1,truck,16712,1.243696e+07,1.315443e+07,2.322978e+06
7,2020,2,rail,9969,1.440743e+06,5.334045e+05,9.368873e+05
8,2020,5,multiple_modes_mail_intermodal_like,17272,6.225655e+05,2.484463e+06,5.400120e+05
9,2021,1,truck,16732,1.272850e+07,1.341445e+07,2.359516e+06



Demand by mode, East-Plus-Gulf:


,year,dms_mode,mode_name,n_rows,tons,value,tmiles
0,2018,1,truck,10746,1.059838e+07,1.065195e+07,1.724385e+06
1,2018,2,rail,6632,1.073775e+06,4.333738e+05,5.027365e+05
2,2018,5,multiple_modes_mail_intermodal_like,10719,4.982136e+05,1.652863e+06,3.089701e+05
3,2019,1,truck,10746,1.051083e+07,1.051107e+07,1.696504e+06
4,2019,2,rail,6595,1.059834e+06,4.349243e+05,4.882757e+05
5,2019,5,multiple_modes_mail_intermodal_like,10710,4.826996e+05,1.630081e+06,2.952965e+05
6,2020,1,truck,10746,1.015901e+07,1.000866e+07,1.639668e+06
7,2020,2,rail,6490,9.820898e+05,3.978007e+05,4.501876e+05
8,2020,5,multiple_modes_mail_intermodal_like,10710,4.561369e+05,1.566443e+06,2.769480e+05
9,2021,1,truck,10745,1.043333e+07,1.021035e+07,1.676780e+06


,faf_orig,faf_dest,dms_mode,mode_name,year,tons,value,tmiles,n_sctg2,n_records,east_plus_gulf_orig_flag,east_plus_gulf_dest_flag,east_plus_gulf_faf_pair_flag
0,011,011,1,truck,2018,35880.873479,15326.185825,1483.641332,29,149,True,True,True
1,011,011,2,rail,2018,2027.859777,14785.299615,128.499054,6,7,True,True,True
2,011,011,5,multiple_modes_mail_intermodal_like,2018,120.107599,1430.297501,6.632377,18,26,True,True,True
3,011,012,1,truck,2018,1003.790981,1260.526660,256.687371,20,329,True,True,True
4,011,012,2,rail,2018,224.597307,211.560396,66.574569,10,118,True,True,True
5,011,012,5,multiple_modes_mail_intermodal_like,2018,42.584159,536.779846,11.718068,8,135,True,True,True
6,011,019,1,truck,2018,10307.348476,8077.511538,1113.272652,28,199,True,True,True
7,011,019,2,rail,2018,21.458409,1.282200,2.499999,1,2,True,True,True
8,011,019,5,multiple_modes_mail_intermodal_like,2018,307.220916,2119.505217,36.004755,14,23,True,True,True
9,011,020,5,multiple_modes_mail_intermodal_like,2018,0.037479,0.233189,0.168731,8,10,True,False,False


,faf_orig,faf_dest,dms_mode,mode_name,year,tons,value,tmiles,n_sctg2,n_records,east_plus_gulf_orig_flag,east_plus_gulf_dest_flag,east_plus_gulf_faf_pair_flag
0,011,011,1,truck,2018,35880.873479,15326.185825,1483.641332,29,149,True,True,True
1,011,011,2,rail,2018,2027.859777,14785.299615,128.499054,6,7,True,True,True
2,011,011,5,multiple_modes_mail_intermodal_like,2018,120.107599,1430.297501,6.632377,18,26,True,True,True
3,011,012,1,truck,2018,1003.790981,1260.526660,256.687371,20,329,True,True,True
4,011,012,2,rail,2018,224.597307,211.560396,66.574569,10,118,True,True,True
5,011,012,5,multiple_modes_mail_intermodal_like,2018,42.584159,536.779846,11.718068,8,135,True,True,True
6,011,019,1,truck,2018,10307.348476,8077.511538,1113.272652,28,199,True,True,True
7,011,019,2,rail,2018,21.458409,1.282200,2.499999,1,2,True,True,True
8,011,019,5,multiple_modes_mail_intermodal_like,2018,307.220916,2119.505217,36.004755,14,23,True,True,True
9,011,050,1,truck,2018,274.187271,236.354833,105.209681,14,24,True,True,True



OK: FAF demand tables are ready.


In [7]:
# ============================================================
# FREIGHT-MNet Processing Step 6:
# Build model-ready supervised edge table
#
# Inputs:
#   1. FMI FAF-pair truck quantile labels:
#      Data\08_processed\fmi_faf_aggregated\
#
#   2. FAF OD demand features:
#      Data\08_processed\graph_inputs\
#
#   3. FAF zone metadata / East-Plus-Gulf flags:
#      Data\09_crosswalks\county_to_faf\east_plus_gulf_faf_zones.parquet
#
# Outputs:
#   Data\08_processed\model_ready\
#     - freight_mnet_supervised_edges_2018_2024_all.parquet
#     - freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
#     - train/val/test split files
#     - feature manifest JSON
#
# Target labels:
#   truck_q25, truck_q50, truck_q75
#
# Features:
#   FAF OD demand by mode:
#     truck / rail / multimodal tons, value, tmiles
#   Demand shares and log transforms
#   Static FAF-zone metadata
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0. Paths
# ------------------------------------------------------------

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")

FMI_AGG_DIR = DATA_ROOT / "08_processed" / "fmi_faf_aggregated"
GRAPH_INPUT_DIR = DATA_ROOT / "08_processed" / "graph_inputs"
CROSSWALK_DIR = DATA_ROOT / "09_crosswalks" / "county_to_faf"

MODEL_READY_DIR = DATA_ROOT / "08_processed" / "model_ready"
MODEL_READY_DIR.mkdir(parents=True, exist_ok=True)

META_DIR = MODEL_READY_DIR / "_metadata"
META_DIR.mkdir(parents=True, exist_ok=True)

# FMI label inputs.
FMI_ALL_PATH = FMI_AGG_DIR / "fmi_faf_pair_quantiles_2018_2024_all.parquet"
FMI_EAST_PATH = FMI_AGG_DIR / "fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.parquet"

# FAF demand inputs.
DEMAND_ALL_PATH = GRAPH_INPUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet"
DEMAND_EAST_PATH = GRAPH_INPUT_DIR / "faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet"

# FAF zone metadata.
FAF_ZONE_META_PATH = CROSSWALK_DIR / "east_plus_gulf_faf_zones.parquet"

# Outputs.
SUP_ALL_PATH = MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_all.parquet"
SUP_ALL_CSV_PREVIEW = MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_all_preview5000.csv"

SUP_EAST_PATH = MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet"
SUP_EAST_CSV_PREVIEW = MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_east_plus_gulf_preview5000.csv"

FEATURE_MANIFEST_PATH = META_DIR / "freight_mnet_supervised_feature_manifest.json"
SUMMARY_PATH = META_DIR / "freight_mnet_supervised_edges_summary.json"

SPLIT_DIR = MODEL_READY_DIR / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = list(range(2018, 2025))

print("=" * 100)
print("Build FREIGHT-MNet supervised edge table")
print("FMI_ALL_PATH:", FMI_ALL_PATH)
print("FMI_EAST_PATH:", FMI_EAST_PATH)
print("DEMAND_ALL_PATH:", DEMAND_ALL_PATH)
print("DEMAND_EAST_PATH:", DEMAND_EAST_PATH)
print("FAF_ZONE_META_PATH:", FAF_ZONE_META_PATH)
print("MODEL_READY_DIR:", MODEL_READY_DIR)
print("=" * 100)

for p in [FMI_ALL_PATH, FMI_EAST_PATH, DEMAND_ALL_PATH, DEMAND_EAST_PATH, FAF_ZONE_META_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required input: {p}")

# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def zfill_faf(s: pd.Series) -> pd.Series:
    out = s.astype("string").str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.str.replace(r"[^0-9]", "", regex=True)
    return out.str.zfill(3)


def safe_div(num, den):
    den = np.asarray(den, dtype=float)
    num = np.asarray(num, dtype=float)
    return np.where(den > 0, num / den, 0.0)


def add_split(year: int) -> str:
    if 2018 <= int(year) <= 2022:
        return "train"
    if int(year) == 2023:
        return "val"
    if int(year) == 2024:
        return "test"
    return "unused"


def load_labels(path: Path, expected_scope: str) -> pd.DataFrame:
    df = pd.read_parquet(path).copy()

    required = [
        "faf_orig",
        "faf_dest",
        "year",
        "truck_q25",
        "truck_q50",
        "truck_q75",
    ]

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Label table missing columns {missing}. Columns={list(df.columns)}")

    df["faf_orig"] = zfill_faf(df["faf_orig"])
    df["faf_dest"] = zfill_faf(df["faf_dest"])
    df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("int16")

    for c in ["truck_q25", "truck_q50", "truck_q75"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[
        df["year"].isin(YEARS)
        & df["truck_q25"].notna()
        & df["truck_q50"].notna()
        & df["truck_q75"].notna()
    ].copy()

    # Ensure monotonic target quantiles.
    q_stack = df[["truck_q25", "truck_q50", "truck_q75"]].to_numpy(dtype=float)
    df["truck_q25"] = np.nanmin(q_stack, axis=1)
    df["truck_q50"] = np.nanmedian(q_stack, axis=1)
    df["truck_q75"] = np.nanmax(q_stack, axis=1)

    # Auxiliary label diagnostics. These are not features.
    df["truck_iqr"] = df["truck_q75"] - df["truck_q25"]
    df["truck_q75_q50_gap"] = df["truck_q75"] - df["truck_q50"]
    df["truck_q50_q25_gap"] = df["truck_q50"] - df["truck_q25"]
    df["truck_iqr_over_q50"] = safe_div(df["truck_iqr"], df["truck_q50"] + 1.0)

    df["scope_from_label"] = expected_scope

    # Ensure one row per FAF OD-year.
    dupes = df.duplicated(["faf_orig", "faf_dest", "year"]).sum()
    if dupes > 0:
        print(f"WARNING: {dupes} duplicate label rows found. Keeping first.")
        df = df.drop_duplicates(["faf_orig", "faf_dest", "year"], keep="first")

    return df


def build_demand_features(path: Path) -> pd.DataFrame:
    demand = pd.read_parquet(path).copy()

    required = [
        "faf_orig",
        "faf_dest",
        "dms_mode",
        "mode_name",
        "year",
        "tons",
        "value",
        "tmiles",
    ]

    missing = [c for c in required if c not in demand.columns]
    if missing:
        raise RuntimeError(f"Demand table missing columns {missing}. Columns={list(demand.columns)}")

    demand["faf_orig"] = zfill_faf(demand["faf_orig"])
    demand["faf_dest"] = zfill_faf(demand["faf_dest"])
    demand["year"] = pd.to_numeric(demand["year"], errors="coerce").astype("int16")
    demand["dms_mode"] = pd.to_numeric(demand["dms_mode"], errors="coerce").astype("int16")

    mode_map = {
        1: "truck",
        2: "rail",
        5: "multimodal",
    }

    demand["mode_key"] = demand["dms_mode"].map(mode_map)

    demand = demand[
        demand["mode_key"].notna()
        & demand["year"].isin(YEARS)
    ].copy()

    for c in ["tons", "value", "tmiles"]:
        demand[c] = pd.to_numeric(demand[c], errors="coerce").fillna(0.0)

    if "n_records" not in demand.columns:
        demand["n_records"] = 1
    if "n_sctg2" not in demand.columns:
        demand["n_sctg2"] = 0

    demand["n_records"] = pd.to_numeric(demand["n_records"], errors="coerce").fillna(0.0)
    demand["n_sctg2"] = pd.to_numeric(demand["n_sctg2"], errors="coerce").fillna(0.0)

    # Re-aggregate defensively.
    demand_agg = (
        demand
        .groupby(["faf_orig", "faf_dest", "year", "mode_key"], as_index=False)
        .agg(
            tons=("tons", "sum"),
            value=("value", "sum"),
            tmiles=("tmiles", "sum"),
            n_records=("n_records", "sum"),
            n_sctg2=("n_sctg2", "max"),
        )
    )

    pivot = demand_agg.pivot_table(
        index=["faf_orig", "faf_dest", "year"],
        columns="mode_key",
        values=["tons", "value", "tmiles", "n_records", "n_sctg2"],
        aggfunc="sum",
        fill_value=0.0,
    )

    # Flatten columns.
    pivot.columns = [f"{metric}_{mode}" for metric, mode in pivot.columns]
    pivot = pivot.reset_index()

    # Ensure all expected mode columns exist.
    modes = ["truck", "rail", "multimodal"]
    metrics = ["tons", "value", "tmiles", "n_records", "n_sctg2"]

    for metric in metrics:
        for mode in modes:
            col = f"{metric}_{mode}"
            if col not in pivot.columns:
                pivot[col] = 0.0

    # Total and shares.
    for metric in ["tons", "value", "tmiles"]:
        truck = pivot[f"{metric}_truck"]
        rail = pivot[f"{metric}_rail"]
        multimodal = pivot[f"{metric}_multimodal"]

        total_col = f"total_{metric}_modes_1_2_5"
        pivot[total_col] = truck + rail + multimodal

        for mode in modes:
            pivot[f"{metric}_share_{mode}"] = safe_div(
                pivot[f"{metric}_{mode}"],
                pivot[total_col],
            )

        # Log transforms.
        for mode in modes:
            pivot[f"log1p_{metric}_{mode}"] = np.log1p(np.maximum(pivot[f"{metric}_{mode}"], 0.0))

        pivot[f"log1p_total_{metric}_modes_1_2_5"] = np.log1p(np.maximum(pivot[total_col], 0.0))

    # Other demand-derived features.
    pivot["has_truck_demand"] = (pivot["tons_truck"] > 0).astype("int8")
    pivot["has_rail_demand"] = (pivot["tons_rail"] > 0).astype("int8")
    pivot["has_multimodal_demand"] = (pivot["tons_multimodal"] > 0).astype("int8")

    pivot["n_modes_with_positive_tons"] = (
        pivot[["has_truck_demand", "has_rail_demand", "has_multimodal_demand"]]
        .sum(axis=1)
        .astype("int8")
    )

    pivot["rail_or_multimodal_tons_share"] = (
        pivot["tons_share_rail"] + pivot["tons_share_multimodal"]
    )

    pivot["truck_to_rail_tons_ratio"] = safe_div(
        pivot["tons_truck"],
        pivot["tons_rail"] + 1e-6,
    )

    pivot["truck_to_multimodal_tons_ratio"] = safe_div(
        pivot["tons_truck"],
        pivot["tons_multimodal"] + 1e-6,
    )

    pivot["same_faf_zone"] = (pivot["faf_orig"] == pivot["faf_dest"]).astype("int8")

    # Clean infinities.
    for c in pivot.columns:
        if pd.api.types.is_numeric_dtype(pivot[c]):
            pivot[c] = pivot[c].replace([np.inf, -np.inf], 0).fillna(0)

    # Ensure one row per OD-year.
    pivot = pivot.drop_duplicates(["faf_orig", "faf_dest", "year"], keep="first").reset_index(drop=True)

    return pivot


def load_zone_meta(path: Path) -> pd.DataFrame:
    meta = pd.read_parquet(path).copy()

    if "faf_zone" not in meta.columns:
        raise RuntimeError(f"FAF zone meta missing faf_zone. Columns={list(meta.columns)}")

    meta["faf_zone"] = zfill_faf(meta["faf_zone"])

    keep = [
        "faf_zone",
        "faf_zone_name",
        "n_counties",
        "n_east_plus_gulf_counties",
        "east_plus_gulf_faf_flag_any_county",
        "east_plus_gulf_county_share",
        "min_county_centroid_lon",
        "max_county_centroid_lon",
    ]

    keep = [c for c in keep if c in meta.columns]
    meta = meta[keep].drop_duplicates("faf_zone").copy()

    # Fill missing numeric / bool fields.
    for c in [
        "n_counties",
        "n_east_plus_gulf_counties",
        "east_plus_gulf_county_share",
        "min_county_centroid_lon",
        "max_county_centroid_lon",
    ]:
        if c in meta.columns:
            meta[c] = pd.to_numeric(meta[c], errors="coerce").fillna(0.0)

    if "east_plus_gulf_faf_flag_any_county" in meta.columns:
        meta["east_plus_gulf_faf_flag_any_county"] = (
            meta["east_plus_gulf_faf_flag_any_county"].fillna(False).astype(bool)
        )

    return meta


def attach_zone_meta(df: pd.DataFrame, zone_meta: pd.DataFrame) -> pd.DataFrame:
    # Origin metadata.
    orig_meta = zone_meta.add_prefix("orig_").rename(columns={"orig_faf_zone": "faf_orig"})
    dest_meta = zone_meta.add_prefix("dest_").rename(columns={"dest_faf_zone": "faf_dest"})

    out = df.merge(orig_meta, on="faf_orig", how="left")
    out = out.merge(dest_meta, on="faf_dest", how="left")

    # Derived static features.
    if "orig_n_counties" in out.columns and "dest_n_counties" in out.columns:
        out["od_n_counties_sum"] = out["orig_n_counties"].fillna(0) + out["dest_n_counties"].fillna(0)
        out["od_n_counties_abs_diff"] = (
            out["orig_n_counties"].fillna(0) - out["dest_n_counties"].fillna(0)
        ).abs()

    if "orig_east_plus_gulf_county_share" in out.columns and "dest_east_plus_gulf_county_share" in out.columns:
        out["od_east_plus_gulf_share_min"] = np.minimum(
            out["orig_east_plus_gulf_county_share"].fillna(0),
            out["dest_east_plus_gulf_county_share"].fillna(0),
        )
        out["od_east_plus_gulf_share_mean"] = (
            out["orig_east_plus_gulf_county_share"].fillna(0)
            + out["dest_east_plus_gulf_county_share"].fillna(0)
        ) / 2.0

    if "orig_min_county_centroid_lon" in out.columns and "dest_min_county_centroid_lon" in out.columns:
        out["od_min_lon_mean"] = (
            out["orig_min_county_centroid_lon"].fillna(0)
            + out["dest_min_county_centroid_lon"].fillna(0)
        ) / 2.0

    return out


def build_supervised_table(label_path: Path, demand_path: Path, zone_meta: pd.DataFrame, scope: str) -> tuple[pd.DataFrame, dict]:
    labels = load_labels(label_path, expected_scope=scope)
    demand_features = build_demand_features(demand_path)

    print("\n" + "=" * 100)
    print(f"Building supervised table: {scope}")
    print("labels rows:", len(labels))
    print("demand feature rows:", len(demand_features))
    print("=" * 100)

    # Merge labels with demand features.
    df = labels.merge(
        demand_features,
        on=["faf_orig", "faf_dest", "year"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    df["demand_merge_status"] = df["_merge"].astype(str)
    df = df.drop(columns=["_merge"])

    n_label_rows_without_demand = int((df["demand_merge_status"] == "left_only").sum())

    # Fill missing demand features with zero.
    demand_feature_cols = [
        c for c in demand_features.columns
        if c not in ["faf_orig", "faf_dest", "year"]
    ]

    for c in demand_feature_cols:
        if c not in df.columns:
            df[c] = 0
        if pd.api.types.is_numeric_dtype(demand_features[c]):
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
        else:
            df[c] = df[c].fillna("missing")

    # Add zone metadata.
    df = attach_zone_meta(df, zone_meta)

    # Split.
    df["split"] = df["year"].apply(add_split)

    # Useful identifiers.
    df["faf_od"] = df["faf_orig"].astype(str) + "->" + df["faf_dest"].astype(str)
    df["scope"] = scope

    # Final feature cleanup.
    for c in df.columns:
        if pd.api.types.is_bool_dtype(df[c]):
            df[c] = df[c].astype("int8")
        elif pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].replace([np.inf, -np.inf], 0).fillna(0)

    # Label checks.
    assert df["truck_q25"].notna().all()
    assert df["truck_q50"].notna().all()
    assert df["truck_q75"].notna().all()
    assert (df["truck_q25"] <= df["truck_q50"]).all()
    assert (df["truck_q50"] <= df["truck_q75"]).all()
    assert df["year"].nunique() == 7

    summary = {
        "scope": scope,
        "label_path": str(label_path),
        "demand_path": str(demand_path),
        "n_rows": int(len(df)),
        "n_unique_faf_od_pairs": int(df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0]),
        "n_years": int(df["year"].nunique()),
        "years": sorted([int(x) for x in df["year"].unique().tolist()]),
        "n_label_rows_without_demand": n_label_rows_without_demand,
        "split_counts": {str(k): int(v) for k, v in df["split"].value_counts().to_dict().items()},
        "demand_merge_status_counts": {str(k): int(v) for k, v in df["demand_merge_status"].value_counts().to_dict().items()},
    }

    return df, summary


def infer_feature_columns(df: pd.DataFrame) -> list[str]:
    label_cols = {
        "truck_q25",
        "truck_q50",
        "truck_q75",
        "truck_iqr",
        "truck_q75_q50_gap",
        "truck_q50_q25_gap",
        "truck_iqr_over_q50",
    }

    id_cols = {
        "scope",
        "scope_from_label",
        "faf_orig",
        "faf_dest",
        "faf_od",
        "year",
        "split",
        "demand_merge_status",
        "orig_faf_zone_name",
        "dest_faf_zone_name",
    }

    # FMI aggregation diagnostics should be available for loss weighting / filtering,
    # but should not be used as predictive features.
    fmi_aux_cols = {
        "n_fmi_county_pairs",
        "obs_weight_sum",
        "obs_weight_mean",
        "obs_weight_max",
        "input_q25_weighted_mean",
        "input_q50_weighted_mean",
        "input_q75_weighted_mean",
        "input_q50_min",
        "input_q50_max",
        "n_orig_counties",
        "n_dest_counties",
    }

    exclude = label_cols | id_cols | fmi_aux_cols

    feature_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            feature_cols.append(c)

    return sorted(feature_cols)


# ------------------------------------------------------------
# 2. Build tables
# ------------------------------------------------------------

zone_meta = load_zone_meta(FAF_ZONE_META_PATH)

all_df, all_summary = build_supervised_table(
    FMI_ALL_PATH,
    DEMAND_ALL_PATH,
    zone_meta,
    scope="all_valid_faf",
)

east_df, east_summary = build_supervised_table(
    FMI_EAST_PATH,
    DEMAND_EAST_PATH,
    zone_meta,
    scope="east_plus_gulf",
)

# Feature columns inferred from all_df; east_df should share them.
feature_cols = infer_feature_columns(all_df)

label_cols = ["truck_q25", "truck_q50", "truck_q75"]
loss_weight_cols = ["n_fmi_county_pairs", "obs_weight_sum"]

# Ensure east_df has all feature columns.
for c in feature_cols:
    if c not in east_df.columns:
        east_df[c] = 0.0

# Sort columns in a clean order.
front_cols = [
    "scope",
    "faf_orig",
    "faf_dest",
    "faf_od",
    "year",
    "split",
    "truck_q25",
    "truck_q50",
    "truck_q75",
    "truck_iqr",
    "truck_iqr_over_q50",
]

aux_cols = [
    c for c in [
        "n_fmi_county_pairs",
        "obs_weight_sum",
        "obs_weight_mean",
        "obs_weight_max",
        "demand_merge_status",
        "orig_faf_zone_name",
        "dest_faf_zone_name",
    ]
    if c in all_df.columns
]

ordered_cols = front_cols + aux_cols + feature_cols
ordered_cols = [c for c in ordered_cols if c in all_df.columns]

# Add any remaining columns at the end.
remaining_cols = [c for c in all_df.columns if c not in ordered_cols]
ordered_cols = ordered_cols + remaining_cols

all_df = all_df[ordered_cols].copy()
east_df = east_df[[c for c in ordered_cols if c in east_df.columns]].copy()

# ------------------------------------------------------------
# 3. Save full tables
# ------------------------------------------------------------

all_df.to_parquet(SUP_ALL_PATH, index=False)
east_df.to_parquet(SUP_EAST_PATH, index=False)

all_df.head(5000).to_csv(SUP_ALL_CSV_PREVIEW, index=False, encoding="utf-8-sig")
east_df.head(5000).to_csv(SUP_EAST_CSV_PREVIEW, index=False, encoding="utf-8-sig")

# Save split files.
split_paths = {}

for scope_name, df in [
    ("all", all_df),
    ("east_plus_gulf", east_df),
]:
    for split_name in ["train", "val", "test"]:
        out_path = SPLIT_DIR / f"freight_mnet_supervised_edges_{scope_name}_{split_name}.parquet"
        df_split = df[df["split"] == split_name].copy()
        df_split.to_parquet(out_path, index=False)
        split_paths[f"{scope_name}_{split_name}"] = str(out_path)

# ------------------------------------------------------------
# 4. Save manifest and summary
# ------------------------------------------------------------

feature_manifest = {
    "feature_columns": feature_cols,
    "label_columns": label_cols,
    "loss_weight_columns": loss_weight_cols,
    "id_columns": ["scope", "faf_orig", "faf_dest", "faf_od", "year", "split"],
    "excluded_fmi_aux_columns": [
        "n_fmi_county_pairs",
        "obs_weight_sum",
        "obs_weight_mean",
        "obs_weight_max",
        "input_q25_weighted_mean",
        "input_q50_weighted_mean",
        "input_q75_weighted_mean",
        "input_q50_min",
        "input_q50_max",
        "n_orig_counties",
        "n_dest_counties",
    ],
    "note": (
        "FMI aggregation diagnostics are saved but excluded from predictive features "
        "to avoid label-derived leakage. They can be used for sample weighting or filtering."
    ),
}

FEATURE_MANIFEST_PATH.write_text(json.dumps(feature_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

summary = {
    "status": "ok",
    "all_summary": all_summary,
    "east_plus_gulf_summary": east_summary,
    "outputs": {
        "supervised_all_parquet": str(SUP_ALL_PATH),
        "supervised_all_preview_csv": str(SUP_ALL_CSV_PREVIEW),
        "supervised_east_plus_gulf_parquet": str(SUP_EAST_PATH),
        "supervised_east_plus_gulf_preview_csv": str(SUP_EAST_CSV_PREVIEW),
        "feature_manifest": str(FEATURE_MANIFEST_PATH),
        "split_paths": split_paths,
    },
    "n_feature_columns": len(feature_cols),
    "label_columns": label_cols,
    "train_years": [2018, 2019, 2020, 2021, 2022],
    "validation_year": 2023,
    "test_year": 2024,
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")

print("\n" + "=" * 100)
print("SUPERVISED EDGE TABLES COMPLETE")
print("=" * 100)
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nFeature columns:")
print(feature_cols)

print("\nAll-scope preview:")
display(all_df.head(20))

print("\nEast-Plus-Gulf preview:")
display(east_df.head(20))

# ------------------------------------------------------------
# 5. Final assertions
# ------------------------------------------------------------

assert len(all_df) > 0
assert len(east_df) > 0
assert all_df["year"].nunique() == 7
assert east_df["year"].nunique() == 7
assert set(all_df["split"].unique()) == {"train", "val", "test"}
assert set(east_df["split"].unique()) == {"train", "val", "test"}
assert (all_df["truck_q25"] <= all_df["truck_q50"]).all()
assert (all_df["truck_q50"] <= all_df["truck_q75"]).all()
assert (east_df["truck_q25"] <= east_df["truck_q50"]).all()
assert (east_df["truck_q50"] <= east_df["truck_q75"]).all()

print("\nOK: model-ready supervised edge tables are ready.")

Build FREIGHT-MNet supervised edge table
FMI_ALL_PATH: E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\fmi_faf_pair_quantiles_2018_2024_all.parquet
FMI_EAST_PATH: E:\NetworkOptimization\pythonProject1\Data\08_processed\fmi_faf_aggregated\fmi_faf_pair_quantiles_2018_2024_east_plus_gulf.parquet
DEMAND_ALL_PATH: E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet
DEMAND_EAST_PATH: E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\faf_od_demand_modes_1_2_5_2018_2024_east_plus_gulf_od_mode_year.parquet
FAF_ZONE_META_PATH: E:\NetworkOptimization\pythonProject1\Data\09_crosswalks\county_to_faf\east_plus_gulf_faf_zones.parquet
MODEL_READY_DIR: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready

Building supervised table: all_valid_faf
labels rows: 112675
demand feature rows: 121705

Building supervised table: east_plus_gulf
labels rows: 73972
demand feature r

,scope,faf_orig,faf_dest,faf_od,year,split,truck_q25,truck_q50,truck_q75,truck_iqr,...,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties,truck_q75_q50_gap,truck_q50_q25_gap,scope_from_label
0,all_valid_faf,011,011,011->011,2018,train,1.00,37.02,71.28,70.28,...,33.156860,47.766945,155.222758,1.00,562.50,11,11,34.26,36.02,all_valid_faf
1,all_valid_faf,011,012,011->012,2018,train,248.18,943.27,1925.53,1677.35,...,516.764447,966.672508,2184.157713,173.83,1979.45,11,2,982.26,695.09,all_valid_faf
2,all_valid_faf,011,019,011->019,2018,train,48.00,100.10,218.95,170.95,...,95.525005,178.619190,492.856903,1.00,1646.70,11,54,118.85,52.10,all_valid_faf
3,all_valid_faf,011,041,011->041,2018,train,1929.44,2498.82,3835.27,1905.83,...,2088.158470,2701.561659,4109.080662,2058.85,4197.91,8,2,1336.45,569.38,all_valid_faf
4,all_valid_faf,011,042,011->042,2018,train,1782.55,2332.43,3435.48,1652.93,...,1980.692788,2536.740261,3894.585715,1972.80,4136.00,8,1,1103.05,549.88,all_valid_faf
5,all_valid_faf,011,049,011->049,2018,train,1860.02,2576.80,3989.00,2128.98,...,1993.514928,2623.732939,4062.570833,1743.69,4108.96,8,8,1412.20,716.78,all_valid_faf
6,all_valid_faf,011,050,011->050,2018,train,404.01,1010.39,1818.68,1414.67,...,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63,808.29,606.38,all_valid_faf
7,all_valid_faf,011,061,011->061,2018,train,2414.93,2921.98,3998.47,1583.54,...,2420.813245,2977.466031,4206.878474,2262.08,4093.48,8,5,1076.49,507.05,all_valid_faf
8,all_valid_faf,011,062,011->062,2018,train,3039.21,4153.81,5892.74,2853.53,...,3398.950625,4425.448750,6813.383125,3125.40,5892.74,4,4,1738.93,1114.60,all_valid_faf
9,all_valid_faf,011,063,011->063,2018,train,2439.78,3273.48,4479.53,2039.75,...,2446.170000,3308.870000,4487.110000,3254.07,3399.06,3,1,1206.05,833.70,all_valid_faf



East-Plus-Gulf preview:


,scope,faf_orig,faf_dest,faf_od,year,split,truck_q25,truck_q50,truck_q75,truck_iqr,...,input_q25_weighted_mean,input_q50_weighted_mean,input_q75_weighted_mean,input_q50_min,input_q50_max,n_orig_counties,n_dest_counties,truck_q75_q50_gap,truck_q50_q25_gap,scope_from_label
0,east_plus_gulf,011,011,011->011,2018,train,1.00,37.02,71.28,70.28,...,33.156860,47.766945,155.222758,1.00,562.50,11,11,34.26,36.02,east_plus_gulf
1,east_plus_gulf,011,012,011->012,2018,train,248.18,943.27,1925.53,1677.35,...,516.764447,966.672508,2184.157713,173.83,1979.45,11,2,982.26,695.09,east_plus_gulf
2,east_plus_gulf,011,019,011->019,2018,train,48.00,100.10,218.95,170.95,...,95.525005,178.619190,492.856903,1.00,1646.70,11,54,118.85,52.10,east_plus_gulf
3,east_plus_gulf,011,050,011->050,2018,train,404.01,1010.39,1818.68,1414.67,...,592.219008,1026.204917,1934.555280,222.58,3217.98,11,63,808.29,606.38,east_plus_gulf
4,east_plus_gulf,011,091,011->091,2018,train,2579.38,3718.82,5127.33,2547.95,...,2510.728667,3559.798000,5716.874000,2557.95,4924.82,8,2,1408.51,1139.44,east_plus_gulf
5,east_plus_gulf,011,092,011->092,2018,train,2242.10,3084.89,4836.23,2594.13,...,2252.316883,3218.952240,5135.916817,2362.98,5337.73,8,4,1751.34,842.79,east_plus_gulf
6,east_plus_gulf,011,099,011->099,2018,train,2307.02,3061.08,4858.83,2551.81,...,2364.197143,3353.897143,5706.734286,2475.53,4970.50,4,2,1797.75,754.06,east_plus_gulf
7,east_plus_gulf,011,101,011->101,2018,train,1731.31,2472.42,3873.20,2141.89,...,1823.161674,2573.021773,3973.259477,2137.94,3250.61,8,2,1400.78,741.11,east_plus_gulf
8,east_plus_gulf,011,121,011->121,2018,train,899.55,1365.26,2482.71,1583.16,...,856.048234,1411.309250,2538.148244,480.71,2541.90,11,6,1117.45,465.71,east_plus_gulf
9,east_plus_gulf,011,122,011->122,2018,train,1306.35,1584.57,2828.86,1522.51,...,1314.750359,1629.469826,2767.383169,1309.43,2242.65,10,7,1244.29,278.22,east_plus_gulf



OK: model-ready supervised edge tables are ready.


In [8]:
from pathlib import Path
import json
import pandas as pd

DATA_ROOT = Path(r"E:\NetworkOptimization\pythonProject1\Data")
MODEL_READY_DIR = DATA_ROOT / "08_processed" / "model_ready"

paths = {
    "supervised_all": MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_all.parquet",
    "supervised_east": MODEL_READY_DIR / "freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet",
    "summary": MODEL_READY_DIR / "_metadata" / "freight_mnet_supervised_edges_summary.json",
    "feature_manifest": MODEL_READY_DIR / "_metadata" / "freight_mnet_supervised_feature_manifest.json",
}

for name, p in paths.items():
    print(name, p.exists(), p, round(p.stat().st_size / 1024**2, 4) if p.exists() else None)

all_df = pd.read_parquet(paths["supervised_all"])
east_df = pd.read_parquet(paths["supervised_east"])
summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
feature_manifest = json.loads(paths["feature_manifest"].read_text(encoding="utf-8"))

print("\nSummary:")
print(json.dumps(summary, indent=2, ensure_ascii=False))

print("\nFeature manifest:")
print("n_features:", len(feature_manifest["feature_columns"]))
print("labels:", feature_manifest["label_columns"])
print("loss weights:", feature_manifest["loss_weight_columns"])

print("\nAll supervised table:")
print("rows:", len(all_df))
print("years:", sorted(all_df["year"].unique().tolist()))
print("splits:", all_df["split"].value_counts().to_dict())
print("unique FAF OD:", all_df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("q monotonic:", bool((all_df["truck_q25"] <= all_df["truck_q50"]).all() and (all_df["truck_q50"] <= all_df["truck_q75"]).all()))
print("demand merge:", all_df["demand_merge_status"].value_counts().to_dict())

print("\nEast-Plus-Gulf supervised table:")
print("rows:", len(east_df))
print("years:", sorted(east_df["year"].unique().tolist()))
print("splits:", east_df["split"].value_counts().to_dict())
print("unique FAF OD:", east_df[["faf_orig", "faf_dest"]].drop_duplicates().shape[0])
print("q monotonic:", bool((east_df["truck_q25"] <= east_df["truck_q50"]).all() and (east_df["truck_q50"] <= east_df["truck_q75"]).all()))
print("demand merge:", east_df["demand_merge_status"].value_counts().to_dict())

print("\nTrain/Val/Test row counts, all:")
display(all_df.groupby(["split", "year"]).size().reset_index(name="n_rows"))

print("\nTrain/Val/Test row counts, East-Plus-Gulf:")
display(east_df.groupby(["split", "year"]).size().reset_index(name="n_rows"))

print("\nFeature sample:")
display(all_df[["faf_orig", "faf_dest", "year", "split", "truck_q25", "truck_q50", "truck_q75"] + feature_manifest["feature_columns"][:10]].head(20))

assert len(all_df) > 0
assert len(east_df) > 0
assert set(all_df["split"].unique()) == {"train", "val", "test"}
assert set(east_df["split"].unique()) == {"train", "val", "test"}
assert (all_df["truck_q25"] <= all_df["truck_q50"]).all()
assert (all_df["truck_q50"] <= all_df["truck_q75"]).all()
assert (east_df["truck_q25"] <= east_df["truck_q50"]).all()
assert (east_df["truck_q50"] <= east_df["truck_q75"]).all()

print("\nOK: supervised model-ready data is ready.")

supervised_all True E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_all.parquet 47.8209
supervised_east True E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet 31.3776
summary True E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_edges_summary.json 0.0036
feature_manifest True E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\_metadata\freight_mnet_supervised_feature_manifest.json 0.0026

Summary:
{
  "status": "ok",
  "all_summary": {
    "scope": "all_valid_faf",
    "label_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\fmi_faf_aggregated\\fmi_faf_pair_quantiles_2018_2024_all.parquet",
    "demand_path": "E:\\NetworkOptimization\\pythonProject1\\Data\\08_processed\\graph_inputs\\faf_od_demand_modes_1_2_5_2018_2024_od_mode_year.parquet",
    "n_rows": 1126

,split,year,n_rows
0,test,2024,16066
1,train,2018,15021
2,train,2019,16346
3,train,2020,16458
4,train,2021,16389
5,train,2022,16238
6,val,2023,16157



Train/Val/Test row counts, East-Plus-Gulf:


,split,year,n_rows
0,test,2024,10574
1,train,2018,9936
2,train,2019,10704
3,train,2020,10761
4,train,2021,10721
5,train,2022,10651
6,val,2023,10625



Feature sample:


,faf_orig,faf_dest,year,split,truck_q25,truck_q50,truck_q75,dest_east_plus_gulf_county_share,dest_east_plus_gulf_faf_flag_any_county,dest_max_county_centroid_lon,dest_min_county_centroid_lon,dest_n_counties,dest_n_east_plus_gulf_counties,has_multimodal_demand,has_rail_demand,has_truck_demand,log1p_tmiles_multimodal
0,011,011,2018,train,1.00,37.02,71.28,1.0,1,-85.797517,-87.297210,11,11,1.0,1.0,1.0,2.032399
1,011,012,2018,train,248.18,943.27,1925.53,1.0,1,-87.722290,-88.205938,2,2,1.0,1.0,1.0,2.543024
2,011,019,2018,train,48.00,100.10,218.95,1.0,1,-85.184930,-88.263310,54,54,1.0,1.0,1.0,3.611046
3,011,041,2018,train,1929.44,2498.82,3835.27,0.0,0,-111.344695,-112.493264,2,0,1.0,0.0,1.0,1.033705
4,011,042,2018,train,1782.55,2332.43,3435.48,0.0,0,-110.846581,-111.789206,2,0,1.0,1.0,1.0,0.024118
5,011,049,2018,train,1860.02,2576.80,3989.00,0.0,0,-109.239951,-113.982672,11,0,1.0,0.0,1.0,0.003351
6,011,050,2018,train,404.01,1010.39,1818.68,1.0,1,-90.052379,-94.274420,75,75,1.0,1.0,1.0,4.642235
7,011,061,2018,train,2414.93,2921.98,3998.47,0.0,0,-115.993876,-119.083380,5,0,1.0,1.0,1.0,3.414265
8,011,062,2018,train,3039.21,4153.81,5892.74,0.0,0,-120.525051,-121.900539,7,0,1.0,0.0,1.0,0.130255
9,011,063,2018,train,2439.78,3273.48,4479.53,0.0,0,-116.734681,-116.734681,1,0,1.0,1.0,1.0,0.292298



OK: supervised model-ready data is ready.
